Resource
-

In [1]:
!pip install wandb==0.22.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.6/19.6 MB 27.1 MB/s eta 0:00:00
  Attempting uninstall: wandb
    Found existing installation: wandb 0.25.1
    Uninstalling wandb-0.25.1:
      Successfully uninstalled wandb-0.25.1


### Install kaleido for export pdf

In [2]:
!pip install plotly==6.4.0
!pip install kaleido==1.2.0
!pip install choreographer==1.2.1
!pip install pypdf
!apt update
!apt-get install libnss3 libatk-bridge2.0-0 libcups2 libxcomposite1 libxdamage1 libxfixes3 libxrandr2 libgbm1 libxkbcommon0 libpango-1.0-0 libcairo2 libasound2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 40.5 MB/s eta 0:00:00
  Attempting uninstall: plotly
    Found existing installation: plotly 5.24.1
    Uninstalling plotly-5.24.1:
      Successfully uninstalled plotly-5.24.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.0/58.0 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.3/49.3 kB 2.2 MB/s eta 0:00:00
  Attempting uninstall: choreographer
    Found existing installation: choreographer 1.4.0
    Uninstalling choreographer-1.4.0:
      Successfully uninstalled choreographer-1.4.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 8.2 MB/s eta 0:00:00
Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 

In [3]:
import plotly.io as pio

pio.get_chrome()

PosixPath('/usr/local/lib/python3.12/dist-packages/choreographer/cli/browser_exe/chrome-linux64/chrome')

# Login

In [1]:
from google.colab import userdata
WANDB_KEY = userdata.get('WANDB_TOKEN')

In [2]:
import wandb
print(wandb.__version__)
wandb.login(key=WANDB_KEY)

/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

0.22.0


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: rofifpriyo (rofifpriyo-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

# Parameter

In [3]:
import wandb
from wandb.apis.public import Api
import pandas as pd
import numpy as np

import plotly.express as px
import plotly.subplots as ps
import plotly.graph_objects as go

ENTITY = wandb.apis.PublicApi().default_entity
PROJECT = "calibration-on-quantized-multilingual"

# 3x Run

## Real 3x Run

In [4]:
pd.options.mode.chained_assignment = None

def visualize_deliver_bar(eval_languages, runs, base_models, quantization_technique, bits, eval_dataset, custom_title="", vertical_spacing=0.1, horizontal_spacing=0.05, display_mean=True, compile=False, random_guess_score=0.25, embed_sinq=True, generate_until=False, strict_answer=False, write_pdf=""):
  language_sort = {
      # Flores Plus
      "eng_Latn": 0, "fra_Latn": 1, "hin_Deva": 2, "npi_Deva": 3, "tam_Taml": 6, "tel_Telu": 7, "ind_Latn": 8, "zsm_Latn": 9, "swh_Latn": 10, "yor_Latn": 11, "cmn_Hans": 12, "jpn_Jpan": 13, "kor_Hang": 14, "arb_Arab" : 15,\
      # Wikipedia
      "en": 0, "fr": 1, "hi": 2, "ne": 3, "ta": 6, "te": 7, "id": 8, "ms": 9, "sw": 10, "yo": 11, "zh": 12, "ja": 13, "ko": 14, "ar": 15,\
      # INCLUDE
      "french": 1, "hindi": 2, "nepali": 3, "albanian": 4, "tamil": 6, "telugu": 7, "indonesian": 8, "malay": 9, "chinese": 12, "japanese": 13, "korean": 14, "arabic": 15,\
      "dutch": 0.2, "german": 0.4, "tagalog": 8.5, "malayalam": 7.5,\
      # Global MMLU Lite
      "en": 0, "fr": 1, "hi": 2, "bn": 5, "id": 8, "sw": 10, "yo": 11, "zh": 12, "ja": 13, "ko": 14, "ar": 15,
      # Multi Blimp
      "eng": 0, "tam": 6, "idn": 8, "swa": 10, "zhoblimp": 12,
      # Belebele
      "eng_Latn": 0, "nld_Latn": 0.2, "deu_Latn": 0.4, "mal_Mlym": 5.8, "tam_Taml": 6, "tel_Telu": 7, "ind_Latn": 8, "tgl_Latn": 8.5, "zsm_Latn": 9, "swh_Latn": 10, "yor_Latn": 11, "lug_Latn": 10.5, "bod_Tibt": 12.2, "mya_Mymr": 12.4, "zho_Hans": 12,
      # Massive
      'en-US': 0, 'id-ID': 8, 'ta-IN': 6, 'sw-KE': 10, 'zh-CN': 12,
      # MGSM
      "en": 0, "ta": 6, "sw": 10, "zh": 12,
  }
  # Add include evals, and count their respective language's family
  family_counter = []
  if eval_dataset.lower() == 'belebele':
    flores_plus_add = ["eng_Latn", "nld_Latn", "deu_Latn"]
    for index, lang in enumerate(flores_plus_add):
      language_sort[lang] = 0 + index / 100
    if not compile:
      family_counter.append(sum([True if i in eval_languages else False for i in flores_plus_add]))
    flores_plus_add = ["mal_Mlym", "tam_Taml", "tel_Telu"]
    for index, lang in enumerate(flores_plus_add):
      language_sort[lang] = 6 + index / 100
    if not compile:
      family_counter.append(sum([True if i in eval_languages else False for i in flores_plus_add]))
    flores_plus_add = ["ind_Latn", "tgl_Latn", "zsm_Latn"]
    for index, lang in enumerate(flores_plus_add):
      language_sort[lang] = 8 + index / 100
    if not compile:
      family_counter.append(sum([True if i in eval_languages else False for i in flores_plus_add]))
    flores_plus_add = ["swh_Latn", "yor_Latn", "lug_Latn"]
    for index, lang in enumerate(flores_plus_add):
      language_sort[lang] = 10 + index / 100
    if not compile:
      family_counter.append(sum([True if i in eval_languages else False for i in flores_plus_add]))
    flores_plus_add = ["bod_Tibt", "mya_Mymr", "zho_Hans"]
    for index, lang in enumerate(flores_plus_add):
      language_sort[lang] = 12 + index / 100
    if not compile:
      family_counter.append(sum([True if i in eval_languages else False for i in flores_plus_add]))
  eval_languages = list(sorted(eval_languages, key=lambda x: language_sort.get(x, 100)))

  columns = ["Base Model", "Quantization Technique", "Bit Width", "Calibration Language"]
  columns.extend(eval_languages)
  custom_sort = {'Unquantized': 0, 'English': 1, 'Tamil': 2, 'Indonesian': 3, 'Swahili': 4, 'Chinese': 5,\
                 'Unquantized': 0, 'gptq': 1, 'slimllm': 2, 'tacq': 3,\
                 "Qwen/Qwen3-8B": 0, "CohereLabs/aya-expanse-8b": 1, "meta-llama/Llama-3.1-8B-Instruct": 2, "Qwen/Qwen3-14B": 3,\
                 "32-bit": 0, "2-bit": 1, "3-bit": 2, "4-bit": 3}

  calib_label_list = ["EN", "TA", "ID", "SW", "ZH"]
  calib_label = {"English": "EN", "Tamil": "TA", "Indonesian": "ID", "Swahili": "SW", "Chinese": "ZH"}
  base_model_label = {"Qwen/Qwen3-8B": "Qwen3", "CohereLabs/aya-expanse-8b": "Aya", "meta-llama/Llama-3.1-8B-Instruct": "Llama", "Qwen/Qwen3-14B": "Qwen3 14B"}
  eval_label = {
      # Global PIQA
      "eng_latn": "EN", "tam_taml": "TA", "ind_latn": "ID", "swh_latn": "SW", "cmn_hans": "ZH",\
      # INCLUDE
      "french": "FR", "hindi": "HI", "nepali": "NE", "albanian": "SQ", "tamil": "TA", "telugu": "TE", "indonesian": "ID", "malay": "MS", "chinese": "ZH", "japanese": "JA", "korean": "KO", "arabic": "AR",\
      "dutch": "NL", "german": "DE", "tagalog": "TL", "malayalam": "ML",\
      "english": "EN",\
      # Global MMLU Lite
      "en": "EN", "fr": "FR", "hi": "HI", "bn": "BN", "id": "ID", "sw": "SW", "yo": "YO", "zh": "ZH", "ja": "JA", "ko": "KO", "ar": "AR",
      # Multi Blimp
      "eng": "EN", "tam": "TA", "idn": "ID", "swa": "SW", "zhoblimp": "ZH",
      # Belebele
      "eng_Latn": "EN", "nld_Latn": "NL", "deu_Latn": "DE", "mal_Mlym": "ML", "tam_Taml": "TA", "tel_Telu": "TE", "ind_Latn": "ID", "tgl_Latn": "TL", "zsm_Latn": "MS", "swh_Latn": "SW", "yor_Latn": "YO", "lug_Latn": "LG", "bod_Tibt": "BO", "mya_Mymr": "MY", "zho_Hans": "ZH",
      # Massive
      'en-US': "EN", 'id-ID': "ID", 'ta-IN': "TA", 'sw-KE': "SW", 'zh-CN': "ZH",
      # MGSM
      "en": "EN", "ta": "TA", "sw": "SW", "zh": "ZH",
  }
  eval_label_list = [eval_label.get(x, x) for x in eval_languages]

  data_gptq = []
  data_slimllm = []
  data_tacq = []
  data_sinq = []
  quantization_techniques = set()

  unquantized_data_acc = []

  # Iterate over the runs and print details
  for run in runs:
      config = run.config
      summary = run.summary

      run_data_acc = []
      run_data_acc.extend([run.config["base_model"], config["quantization_technique"], config["bit_width"], config.get("calibration_language", None).capitalize() if config.get("calibration_language", None) else None])
      for lang in eval_languages:
        if not generate_until:
          run_data_acc.append(run.summary.get(f"{eval_dataset.lower()}_acc_{lang}", None))
        else:
          strict = run.summary.get(f"{eval_dataset.lower()}_strict_{lang}", None)
          flexible = run.summary.get(f"{eval_dataset.lower()}_flexible_{lang}", None)
          run_data_acc.append(max(strict, flexible) if not strict_answer else strict)

      quantization_techniques.add(config["quantization_technique"])
      if config["quantization_technique"] == "Unquantized":
        unquantized_data_acc.append(run_data_acc)
      elif config["quantization_technique"] == "gptq":
        data_gptq.append(run_data_acc)
      elif config["quantization_technique"] == "slimllm":
        data_slimllm.append(run_data_acc)
      elif config["quantization_technique"] == "tacq":
        data_tacq.append(run_data_acc)
      elif config["quantization_technique"] == "sinq":
        data_sinq.append(run_data_acc)

  df_unquantized_acc_chief = pd.DataFrame(unquantized_data_acc, columns=columns)
  df_gptq_chief = pd.DataFrame(data_gptq, columns=columns).sort_values(by=['Base Model', 'Quantization Technique', 'Calibration Language', "Bit Width"], key=lambda x: x.map(custom_sort))
  df_slimllm_chief = pd.DataFrame(data_slimllm, columns=columns).sort_values(by=['Base Model', 'Quantization Technique', 'Calibration Language', "Bit Width"], key=lambda x: x.map(custom_sort))
  df_tacq_chief = pd.DataFrame(data_tacq, columns=columns).sort_values(by=['Base Model', 'Quantization Technique', 'Calibration Language', "Bit Width"], key=lambda x: x.map(custom_sort))
  df_sinq_chief = pd.DataFrame(data_sinq, columns=columns).sort_values(by=['Base Model', "Bit Width"], key=lambda x: x.map(custom_sort))

  print("Pre-AGG")
  display(df_gptq_chief)
  display(df_slimllm_chief)
  display(df_tacq_chief)

  if eval_dataset.lower() == "include" and not compile and 'telugu' not in eval_languages:
    for df_acc in [df_gptq_chief, df_slimllm_chief, df_tacq_chief, df_sinq_chief, df_unquantized_acc_chief]:
      df_acc.insert(5, 'english', (df_acc['german'] + df_acc['dutch']) / (2))
      df_acc.drop(["german", "dutch"], axis=1, inplace=True)

    replace_index = min(eval_languages.index("german"), eval_languages.index("dutch"))
    eval_languages.insert(replace_index, "english")
    eval_languages.remove("german")
    eval_languages.remove("dutch")
    print(eval_label_list)
    eval_label_list.insert(replace_index, "EN")
    eval_label_list.remove("DE")
    eval_label_list.remove("NL")

  # AGG to produce mean and stderr
  df_unquantized_acc_chief_std = df_unquantized_acc_chief.groupby(['Base Model', 'Quantization Technique', 'Calibration Language', 'Bit Width'], as_index=False).agg("std")

  df_gptq_chief_std = df_gptq_chief.groupby(['Base Model', 'Quantization Technique', 'Calibration Language', 'Bit Width'], as_index=False).agg("std", ddof=0).sort_values(by=['Base Model', 'Quantization Technique', 'Calibration Language'], key=lambda x: x.map(custom_sort))
  df_gptq_chief = df_gptq_chief.groupby(['Base Model', 'Quantization Technique', 'Calibration Language', 'Bit Width'], as_index=False).agg("mean").sort_values(by=['Base Model', 'Quantization Technique', 'Calibration Language'], key=lambda x: x.map(custom_sort))

  df_slimllm_chief_std = df_slimllm_chief.groupby(['Base Model', 'Quantization Technique', 'Calibration Language', 'Bit Width'], as_index=False).agg("std", ddof=0).sort_values(by=['Base Model', 'Quantization Technique', 'Calibration Language'], key=lambda x: x.map(custom_sort))
  df_slimllm_chief = df_slimllm_chief.groupby(['Base Model', 'Quantization Technique', 'Calibration Language', 'Bit Width'], as_index=False).agg("mean").sort_values(by=['Base Model', 'Quantization Technique', 'Calibration Language'], key=lambda x: x.map(custom_sort))

  df_tacq_chief_std = df_tacq_chief.groupby(['Base Model', 'Quantization Technique', 'Calibration Language', 'Bit Width'], as_index=False).agg("std", ddof=0).sort_values(by=['Base Model', 'Quantization Technique', 'Calibration Language'], key=lambda x: x.map(custom_sort))
  df_tacq_chief = df_tacq_chief.groupby(['Base Model', 'Quantization Technique', 'Calibration Language', 'Bit Width'], as_index=False).agg("mean").sort_values(by=['Base Model', 'Quantization Technique', 'Calibration Language'], key=lambda x: x.map(custom_sort))

  print("Contrasting Acc and Stderr")
  print("GPTQ")
  display(df_gptq_chief)
  display(df_gptq_chief_std[((df_gptq_chief_std["Bit Width"] == "2-bit") & (df_gptq_chief_std["Base Model"].isin(["Qwen/Qwen3-8B", "CohereLabs/aya-expanse-8b"]))) | ((df_gptq_chief_std["Bit Width"] == "3-bit") & (df_gptq_chief_std["Base Model"] == "Qwen/Qwen3-8B"))])
  print("SLIM")
  display(df_slimllm_chief)
  display(df_slimllm_chief_std[((df_slimllm_chief_std["Bit Width"] == "2-bit") & (df_slimllm_chief_std["Base Model"].isin(["Qwen/Qwen3-8B", "CohereLabs/aya-expanse-8b"]))) | ((df_slimllm_chief_std["Bit Width"] == "3-bit") & (df_slimllm_chief_std["Base Model"] == "Qwen/Qwen3-8B"))])
  print("TACQ")
  display(df_tacq_chief)
  display(df_tacq_chief_std[((df_gptq_chief_std["Bit Width"] == "2-bit") & (df_tacq_chief_std["Base Model"].isin(["Qwen/Qwen3-8B", "CohereLabs/aya-expanse-8b"]))) | ((df_tacq_chief_std["Bit Width"] == "3-bit") & (df_tacq_chief_std["Base Model"] == "Qwen/Qwen3-8B"))])

  # Add figure placeholder before iterations
  from plotly.subplots import make_subplots
  big_fig = make_subplots(rows=len(base_models), cols=len(eval_languages), subplot_titles=tuple([eval_label[lang] for lang in eval_languages]),
                          vertical_spacing=vertical_spacing,
                          horizontal_spacing=horizontal_spacing)

  # Iterate for each model for decoupling
  df_chief = pd.DataFrame([])
  df_zscore_chief = pd.DataFrame([])
  index_border_chief = [0]
  for base_model in base_models:
    df_gptq = df_gptq_chief.loc[df_gptq_chief["Base Model"] == base_model]
    df_gptq_std = df_gptq_chief_std.loc[df_gptq_chief_std["Base Model"] == base_model]
    df_slimllm = df_slimllm_chief.loc[df_slimllm_chief["Base Model"] == base_model]
    df_slimllm_std = df_slimllm_chief_std.loc[df_slimllm_chief_std["Base Model"] == base_model]
    df_tacq = df_tacq_chief.loc[df_tacq_chief["Base Model"] == base_model]
    df_tacq_std = df_tacq_chief_std.loc[df_tacq_chief_std["Base Model"] == base_model]
    df_sinq = df_sinq_chief.loc[df_sinq_chief["Base Model"] == base_model]
    df_unquantized_acc = df_unquantized_acc_chief.loc[df_unquantized_acc_chief["Base Model"] == base_model]
    df_unquantized_acc_std = df_unquantized_acc_chief_std.loc[df_unquantized_acc_chief_std["Base Model"] == base_model]
    base_model = base_model_label[base_model]

    df_gptq.iloc[:, 4:] = df_gptq.iloc[:, 4:].rename(lambda x: eval_label.get(x, x), axis='columns')
    # df_gptq = df_gptq.rename(lambda x: eval_label.get(x, x), axis='columns')
    df_gptq.loc[:, "Calibration Language"] = df_gptq.loc[:, "Calibration Language"].apply(lambda x: calib_label.get(x, x))
    if display_mean:
      df_gptq = pd.concat([df_gptq.T, df_gptq.mean(axis=0).to_frame('Mean')], axis=1).T
    df_gptq[df_gptq.columns[4:]] = df_gptq[df_gptq.columns[4:]].astype(float)
    df_gptq_std.iloc[:, 4:] = df_gptq_std.iloc[:, 4:].rename(lambda x: eval_label.get(x, x), axis='columns')
    df_gptq_std.loc[:, "Calibration Language"] = df_gptq_std.loc[:, "Calibration Language"].apply(lambda x: calib_label.get(x, x))

    df_slimllm.iloc[:, 4:] = df_slimllm.iloc[:, 4:].rename(lambda x: eval_label.get(x, x), axis='columns')
    # df_slimllm = df_slimllm.rename(lambda x: eval_label.get(x, x), axis='columns')
    df_slimllm.loc[:, "Calibration Language"] = df_slimllm.loc[:, "Calibration Language"].apply(lambda x: calib_label.get(x, x))
    if display_mean:
      df_slimllm = pd.concat([df_slimllm.T, df_slimllm.mean(axis=0).to_frame('Mean')], axis=1).T
    df_slimllm[df_gptq.columns[4:]] = df_slimllm[df_slimllm.columns[4:]].astype(float)
    df_slimllm_std.iloc[:, 4:] = df_slimllm_std.iloc[:, 4:].rename(lambda x: eval_label.get(x, x), axis='columns')
    df_slimllm_std.loc[:, "Calibration Language"] = df_slimllm_std.loc[:, "Calibration Language"].apply(lambda x: calib_label.get(x, x))

    df_tacq.iloc[:, 4:] = df_tacq.iloc[:, 4:].rename(lambda x: eval_label.get(x, x), axis='columns')
    # df_tacq = df_tacq.rename(lambda x: eval_label.get(x, x), axis='columns')
    df_tacq.loc[:, "Calibration Language"] = df_tacq.loc[:, "Calibration Language"].apply(lambda x: calib_label.get(x, x))
    if display_mean:
      df_tacq = pd.concat([df_tacq.T, df_tacq.mean(axis=0).to_frame('Mean')], axis=1).T
    df_tacq[df_gptq.columns[4:]] = df_tacq[df_tacq.columns[4:]].astype(float)
    df_tacq_std.iloc[:, 4:] = df_tacq_std.iloc[:, 4:].rename(lambda x: eval_label.get(x, x), axis='columns')
    df_tacq_std.loc[:, "Calibration Language"] = df_tacq_std.loc[:, "Calibration Language"].apply(lambda x: calib_label.get(x, x))

    df_sinq.iloc[:, 4:] = df_sinq.iloc[:, 4:].rename(lambda x: eval_label.get(x, x), axis='columns')
    # df_sinq = df_sinq.rename(lambda x: eval_label.get(x, x), axis='columns')
    df_sinq.loc[:, "Calibration Language"] = df_sinq.loc[:, "Calibration Language"].apply(lambda x: calib_label.get(x, x))
    if display_mean:
      df_sinq = pd.concat([df_tacq.T, df_sinq.mean(axis=0).to_frame('Mean')], axis=1).T
    df_sinq[df_gptq.columns[4:]] = df_sinq[df_sinq.columns[4:]].astype(float)

    # df_unquantized_acc.iloc[:, 4:] = df_unquantized_acc.iloc[:, 4:]
    df_unquantized_acc = df_unquantized_acc.rename(lambda x: eval_label.get(x, x), axis='columns')
    if display_mean:
      df_unquantized_acc = pd.concat([df_unquantized_acc, df_unquantized_acc.mean(axis=0).to_frame('Mean')], axis=0)
    df = pd.concat([df_gptq, df_slimllm, df_tacq], axis=1)
    if display_mean:
      df_mean_quantized = df.mean(axis=1).to_frame('Mean of Quantized')
    df_unquantized_acc_std.iloc[:, 4:] = df_unquantized_acc_std.iloc[:, 4:].rename(lambda x: eval_label.get(x, x), axis='columns')

    # The df structure: Column the eval, index (5 first columns) is the settings
    # Calculate Z-Score
    from scipy.stats import zscore
    import numpy as np
    ## Placeholder for Z-score, Direct, Cross
    df_gptq["zscore"] = np.empty(len(df_gptq))
    df_slimllm["zscore"] = np.empty(len(df_slimllm))
    df_tacq["zscore"] = np.empty(len(df_tacq))

    df_gptq["direct_transposed"] = np.empty(len(df_gptq))
    df_slimllm["direct_transposed"] = np.empty(len(df_slimllm))
    df_tacq["direct_transposed"] = np.empty(len(df_tacq))
    df_gptq["cross_transposed"] = np.empty(len(df_gptq))
    df_slimllm["cross_transposed"] = np.empty(len(df_slimllm))
    df_tacq["cross_transposed"] = np.empty(len(df_tacq))

    df_gptq_std["direct_transposed"] = np.empty(len(df_gptq_std))
    df_slimllm_std["direct_transposed"] = np.empty(len(df_slimllm_std))
    df_tacq_std["direct_transposed"] = np.empty(len(df_tacq_std))
    df_gptq_std["cross_transposed"] = np.empty(len(df_gptq_std))
    df_slimllm_std["cross_transposed"] = np.empty(len(df_slimllm_std))
    df_tacq_std["cross_transposed"] = np.empty(len(df_tacq_std))
    ## Calculate Z-score as usual
    ## Calculate Direct as En -> EN, while Cross as mean of ~En -> EN
    for bit in bits:
      # Acc GPTQ
      df_filter = df_gptq[df_gptq["Bit Width"] == bit]
      if len(df_filter) > 0:
        df_gptq.loc[df_filter.index, "zscore"] = zscore(df_filter.iloc[:, 4:], axis=0)[np.arange(len(df_filter)), [x for x in df_filter["Calibration Language"].apply(lambda x: eval_label_list.index(calib_label.get(x, x))).values.tolist()]]
        mean_ppl = df_filter.iloc[:, 4:4+len(eval_languages)].sum(axis=0, min_count=0)                                    # Seek the column using iloc, because seeking using loc is harder
        df_gptq.loc[df_filter.index, "direct_transposed"] = [df_filter.iloc[i, 4 + eval_label_list.index(calib)] for i, calib in enumerate(df_filter["Calibration Language"].unique())]
        df_gptq.loc[df_filter.index, "cross_transposed"] = (mean_ppl.values - df_gptq.loc[df_filter.index, "direct_transposed"].values) / (len(df_filter) - 1)
        print(list([calib_label_list.index(calib), calib] for i, calib in enumerate(df_filter["Calibration Language"].unique())))

      # Std GPTQ
      df_filter = df_gptq_std[df_gptq_std["Bit Width"] == bit]
      if len(df_filter) > 0:
        df_gptq_std.loc[df_filter.index, "zscore"] = zscore(df_filter.iloc[:, 4:], axis=0)[np.arange(len(df_filter)), [x for x in df_filter["Calibration Language"].apply(lambda x: eval_label_list.index(calib_label.get(x, x))).values.tolist()]]
        mean_ppl = df_filter.iloc[:, 4:4+len(eval_languages)].map(lambda x: np.square(x)).sum(axis=0, min_count=0)                                    # Seek the column using iloc, because seeking using loc is harder
        df_gptq_std.loc[df_filter.index, "direct_transposed"] = [df_filter.iloc[i, 4 + eval_label_list.index(calib)] for i, calib in enumerate(df_filter["Calibration Language"].unique())]
        df_gptq_std.loc[df_filter.index, "cross_transposed"] = np.sqrt((mean_ppl.values - df_gptq_std.loc[df_filter.index, "direct_transposed"].map(lambda x: np.square(x)).values) / (len(df_filter) - 1))

      # Acc SliMLLM
      df_filter = df_slimllm[df_slimllm["Bit Width"] == bit]
      print("where TA located")
      print(df_filter["Calibration Language"].unique())
      if len(df_filter) > 0:
        df_slimllm.loc[df_filter.index, "zscore"] = zscore(df_filter.iloc[:, 4:], axis=0)[np.arange(len(df_filter)), [x for x in df_filter["Calibration Language"].apply(lambda x: eval_label_list.index(calib_label.get(x, x))).values.tolist()]]
        mean_ppl = df_filter.iloc[:, 4:4+len(eval_languages)].sum(axis=0, min_count=0)
        print("mean_ppl")
        display(mean_ppl)
        if base_model == "Aya" and "TA" in eval_label_list:   # Short hand by the renaming indexes before
          mean_ppl = mean_ppl.drop(eval_languages[eval_label_list.index("TA")], axis=0, errors='ignore')                        # Seek the column using iloc, because seeking using loc is harder
        df_slimllm.loc[df_filter.index, "direct_transposed"] = [df_filter.iloc[i, 4 + eval_label_list.index(calib)] for i, calib in enumerate(df_filter["Calibration Language"].unique())]
        df_slimllm.loc[df_filter.index, "cross_transposed"] = (mean_ppl.values - df_slimllm.loc[df_filter.index, "direct_transposed"].values) / (len(df_filter) - 1)

      # Std SliMLLM
      df_filter = df_slimllm_std[df_slimllm_std["Bit Width"] == bit]
      if len(df_filter) > 0:
        df_slimllm_std.loc[df_filter.index, "zscore"] = zscore(df_filter.iloc[:, 4:], axis=0)[np.arange(len(df_filter)), [x for x in df_filter["Calibration Language"].apply(lambda x: eval_label_list.index(calib_label.get(x, x))).values.tolist()]]
        mean_ppl = df_filter.iloc[:, 4:4+len(eval_languages)].map(lambda x: np.square(x)).sum(axis=0, min_count=0)
        if base_model == "Aya" and "TA" in eval_label_list:   # Short hand by the renaming indexes before
          mean_ppl = mean_ppl.drop(eval_languages[eval_label_list.index("TA")], axis=0, errors='ignore')                        # Seek the column using iloc, because seeking using loc is harder
        df_slimllm_std.loc[df_filter.index, "direct_transposed"] = [df_filter.iloc[i, 4 + eval_label_list.index(calib)] for i, calib in enumerate(df_filter["Calibration Language"].unique())]
        df_slimllm_std.loc[df_filter.index, "cross_transposed"] = np.sqrt((mean_ppl.values - df_slimllm_std.loc[df_filter.index, "direct_transposed"].map(lambda x: np.square(x)).values) / (len(df_filter) - 1))

      # Acc TaCQ
      df_filter = df_tacq[df_tacq["Bit Width"] == bit]
      if len(df_filter) > 0:
        df_tacq.loc[df_filter.index, "zscore"] = zscore(df_filter.iloc[:, 4:], axis=0)[np.arange(len(df_filter)), [x for x in df_filter["Calibration Language"].apply(lambda x: eval_label_list.index(calib_label.get(x, x))).values.tolist()]]
        mean_ppl = df_filter.iloc[:, 4:4+len(eval_languages)].sum(axis=0, min_count=0)                                   # Seek the column using iloc, because seeking using loc is harder
        df_tacq.loc[df_filter.index, "direct_transposed"] = [df_filter.iloc[i, 4 + eval_label_list.index(calib)] for i, calib in enumerate(df_filter["Calibration Language"].unique())]
        df_tacq.loc[df_filter.index, "cross_transposed"] = (mean_ppl.values - df_tacq.loc[df_filter.index, "direct_transposed"].values) / (len(df_filter) - 1)

      # Std TaCQ
      df_filter = df_tacq_std[df_tacq_std["Bit Width"] == bit]
      if len(df_filter) > 0:
        df_tacq_std.loc[df_filter.index, "zscore"] = zscore(df_filter.iloc[:, 4:], axis=0)[np.arange(len(df_filter)), [x for x in df_filter["Calibration Language"].apply(lambda x: eval_label_list.index(calib_label.get(x, x))).values.tolist()]]
        mean_ppl = df_filter.iloc[:, 4:4+len(eval_languages)].map(lambda x: np.square(x)).sum(axis=0, min_count=0)                                   # Seek the column using iloc, because seeking using loc is harder
        df_tacq_std.loc[df_filter.index, "direct_transposed"] = [df_filter.iloc[i, 4 + eval_label_list.index(calib)] for i, calib in enumerate(df_filter["Calibration Language"].unique())]
        df_tacq_std.loc[df_filter.index, "cross_transposed"] = np.sqrt((mean_ppl.values - df_tacq_std.loc[df_filter.index, "direct_transposed"].map(lambda x: np.square(x)).values) / (len(df_filter) - 1))

    # display(df_gptq)
    # display(df_slimllm)
    # display(df_tacq)
    # Append to the subplot for each model, languages
    for lang in eval_languages:
      # fig = go.Figure()
      trace = []
      # GPTQ
      for quant in ["sinq", "tacq", "slimllm", "gptq"]:
        if quant == 'sinq':
          df_filter = df_sinq
          marker_pattern_shape = '+'
          # line_dash = 'solid'
          # zorder=0
          shift = -0.2375 - 0.11875 if "slimllm" in quantization_techniques else -0.95/3
        elif quant == 'gptq':
          df_filter = df_gptq.loc[df_gptq["Calibration Language"] == eval_label[lang]]
          df_filter_std = df_gptq_std.loc[df_gptq_std["Calibration Language"] == eval_label[lang]]
          marker_pattern_shape = '/'
          # line_dash = 'solid'
          # zorder=0
          shift = -0.11875 if "slimllm" in quantization_techniques else 0
        elif quant == 'slimllm':
          df_filter = df_slimllm.loc[df_slimllm["Calibration Language"] == eval_label[lang]]
          df_filter_std = df_slimllm_std.loc[df_slimllm_std["Calibration Language"] == eval_label[lang]]
          marker_pattern_shape = '-'
          line_dash = 'dash'
          zorder=1
          shift = 0.11875 if "slimllm" in quantization_techniques else 0
        elif quant == 'tacq':
          df_filter = df_tacq.loc[df_tacq["Calibration Language"] == eval_label[lang]]
          df_filter_std = df_tacq_std.loc[df_tacq_std["Calibration Language"] == eval_label[lang]]
          marker_pattern_shape = '\\'
          line_dash = 'longdash'
          zorder=2
          shift = 0.2375 + 0.11875 if "slimllm" in quantization_techniques else 0.95/3

        # Categorical map for bit width to x-axis in integer
        categorical_map = {}
        for i, bit in enumerate(df_filter["Bit Width"]):
          categorical_map[bit] = i

        if quant != "sinq":
          # print(f"{base_model}, {quant}")
          # display(df_filter_std["cross_transposed"])
          # display(df_filter_std["cross_transposed"].map(lambda x: x != 0))
          df_error_bar = df_filter_std.copy().loc[:, ["Bit Width", "cross_transposed", "direct_transposed"]]
          zero_error = df_error_bar[df_error_bar['direct_transposed'] == 0]
          nonzero_error = df_error_bar[df_error_bar['direct_transposed'] != 0]
          # Without Error Bar
          trace = (go.Bar(x=list(map(lambda x: categorical_map[x] + shift, df_filter["Bit Width"].drop(nonzero_error.index))), y=df_filter["direct_transposed"].drop(nonzero_error.index),
                              # mode='lines+markers',
                              # marker_pattern_shape=marker_pattern_shape,
                              # marker_line_color=marker_line_color,
                              marker_color=px.colors.qualitative.Plotly[0],
                              # marker_line_width=2,
                              # marker_size=5,
                              # line_dash=line_dash,
                              width=([0.2375] if "slimllm" in quantization_techniques else [0.95 / 3]) * len(df_filter["Bit Width"]),
                              # opacity=0.8,
                              # zorder=zorder,
                              showlegend=False,
                              name=f'{quant}-Direct',
                              # text=df_filter["Quantization Technique"] + "-Direct"
                            ))
          big_fig.append_trace(
              trace,
              row=1 + [base_model_label[x] for x in base_models].index(base_model), col=1 + eval_languages.index(lang)
          )
          trace = (go.Bar(x=list(map(lambda x: categorical_map[x] + shift, df_filter["Bit Width"].drop(nonzero_error.index))), y=df_filter["cross_transposed"].drop(nonzero_error.index),
                              # mode='lines+markers',
                              # marker_pattern_shape=marker_pattern_shape,
                              # marker_line_color=marker_line_color,
                              marker_color=px.colors.qualitative.Plotly[4],
                              # marker_line_width=2,
                              # marker_size=5,
                              # line_dash=line_dash,
                              width=([0.11875] if "slimllm" in quantization_techniques else [0.95 / 6]) * len(df_filter["Bit Width"]),
                              # opacity=0.8,
                              # zorder=max(0, 1-zorder),
                              showlegend=False,
                              name=f'{quant}-Cross',
                              # text=df_filter["Quantization Technique"] + "-Cross"
                            ))
          big_fig.append_trace(
              trace,
              row=1 + [base_model_label[x] for x in base_models].index(base_model), col=1 + eval_languages.index(lang)
          )
          # With Error Bar
          trace = (go.Bar(x=list(map(lambda x: categorical_map[x] + shift, df_filter["Bit Width"].drop(zero_error.index))), y=df_filter["direct_transposed"].drop(zero_error.index),
                              error_y=dict(
                                  type='data',
                                  array=df_error_bar["direct_transposed"].drop(zero_error.index),             # Assigns standard deviation array
                                  visible=True,
                                  thickness=4,               # Line width of error bar
                                  width=6,                    # Width of the horizontal cap line
                                  color=px.colors.qualitative.Plotly[5],
                              ),
                              # mode='lines+markers',
                              # marker_pattern_shape=marker_pattern_shape,
                              # marker_line_color=marker_line_color,
                              marker_color=px.colors.qualitative.Plotly[0],
                              # marker_line_width=2,
                              # marker_size=5,
                              # line_dash=line_dash,
                              width=([0.2375] if "slimllm" in quantization_techniques else [0.95 / 3]) * len(df_filter["Bit Width"]),
                              # opacity=0.8,
                              # zorder=zorder,
                              showlegend=False,
                              name=f'{quant}-Direct',
                              # text=df_filter["Quantization Technique"] + "-Direct"
                            ))
          big_fig.append_trace(
              trace,
              row=1 + [base_model_label[x] for x in base_models].index(base_model), col=1 + eval_languages.index(lang)
          )
          trace = (go.Bar(x=list(map(lambda x: categorical_map[x] + shift, df_filter["Bit Width"].drop(zero_error.index))), y=df_filter["cross_transposed"].drop(zero_error.index),
                              error_y=dict(
                                  type='data',
                                  array=df_filter_std["cross_transposed"].drop(zero_error.index),             # Assigns standard deviation array
                                  visible=True,
                                  thickness=4,               # Line width of error bar
                                  width=6,                    # Width of the horizontal cap line
                                  color=px.colors.qualitative.Plotly[9],
                              ),
                              # mode='lines+markers',
                              # marker_pattern_shape=marker_pattern_shape,
                              # marker_line_color=marker_line_color,
                              marker_color=px.colors.qualitative.Plotly[4],
                              # marker_line_width=2,
                              # marker_size=5,
                              # line_dash=line_dash,
                              width=([0.11875] if "slimllm" in quantization_techniques else [0.95 / 6]) * len(df_filter["Bit Width"]),
                              # opacity=0.8,
                              # zorder=max(0, 1-zorder),
                              showlegend=False,
                              name=f'{quant}-Cross',
                              # text=df_filter["Quantization Technique"] + "-Cross"
                            ))
          big_fig.append_trace(
              trace,
              row=1 + [base_model_label[x] for x in base_models].index(base_model), col=1 + eval_languages.index(lang)
          )
        else: # sinq
            # print(f"SINQ: {df_filter[lang]}")
            trace = (go.Bar(x=list(map(lambda x: categorical_map[x] + shift, df_filter["Bit Width"])), y=df_filter[lang],
                              # mode='lines+markers',
                              # marker_pattern_shape=marker_pattern_shape,
                              # marker_line_color=marker_line_color,
                              marker_color=px.colors.qualitative.Plotly[2],
                              # marker_line_width=2,
                              # marker_size=5,
                              # line_dash=line_dash,
                              width=([0.2375] if "slimllm" in quantization_techniques else [0.95 / 3]) * len(df_filter["Bit Width"]),
                              # opacity=0.8,
                              # zorder=zorder,
                              showlegend=False,
                              name=f'{quant}',
                              # text=df_filter["Quantization Technique"] + "-Direct"
                            ))
            big_fig.append_trace(
                trace,
                row=1 + [base_model_label[x] for x in base_models].index(base_model), col=1 + eval_languages.index(lang)
            )

        # Add Quantization method label
        for x_coor in list(map(lambda x: categorical_map[x] + shift, df_filter["Bit Width"])):
          if quant != "sinq":
            big_fig.add_shape(
              type="rect",
              x0=x_coor,
              y0=0,
              x1=x_coor,
              y1=random_guess_score,
              xref=f"x{len(eval_languages) * [base_model_label[x] for x in base_models].index(base_model) + 1 + eval_languages.index(lang)}",
              yref=f"y{len(eval_languages) * [base_model_label[x] for x in base_models].index(base_model) + 1 + eval_languages.index(lang)}",
              line_width=0,
              layer="above",
              label=dict(text=quant[0].upper(), textposition="bottom center", font=dict(size=24)),
            )
        big_fig.update_xaxes(
            # yaxis={'title' : "ACC (Higher is better)"},
            # xaxis = dict(
            tickmode = 'array',
            tickvals = list(range(len(df_filter["Bit Width"]))),
            ticktext = df_filter["Bit Width"],
            # ),
            row=1 + [base_model_label[x] for x in base_models].index(base_model),
            col=1 + eval_languages.index(lang)
        )
      for quant in ["Unquantized"]:
        marker_pattern_shape = ''
        big_fig.add_hline(y=df_unquantized_acc[eval_label.get(lang, lang)].iloc[0], line_width=4, row=1 + [base_model_label[x] for x in base_models].index(base_model), col=1 + eval_languages.index(lang))
        big_fig.add_hline(y=random_guess_score, line_width=4, row=1 + [base_model_label[x] for x in base_models].index(base_model), col=1 + eval_languages.index(lang),
              line_dash="dot",
              # annotation_text="Random Guess Score",
              # annotation_position="bottom right",
        )
        # display(df_unquantized_acc)
        # print(list(filter(lambda x: x < (df_unquantized_acc[eval_label.get(lang, lang)].iloc[0] + 0.20), [0.0, 0.20, 0.40, 0.60, 0.80, 1.0])))
        big_fig.update_yaxes(tickvals=list(filter(lambda x: x < (df_unquantized_acc[eval_label.get(lang, lang)].iloc[0] + 0.20), [0.0, 0.20, 0.40, 0.60, 0.80, 1.0])), row=1 + [base_model_label[x] for x in base_models].index(base_model), col=1 + eval_languages.index(lang))

  # Figure Style
  big_fig.update_xaxes(title_standoff=15, automargin=True)
  big_fig.update_yaxes(title_standoff=5, automargin=True)
  big_fig.update_layout(
      margin=dict(t=100, b=20 if not write_pdf else 35, l=50, r=0)
  )
  # Early Config: t=100, b=15 if not write_pdf else 25, l=50, r=0
  # Early Config: big_fig.layout.height = 100 + 80 * 3 + (15 if not write_pdf else 25)
  # Early Config: big_fig.layout.width = 50 + 160 * len(eval_languages)
  # big_fig.update_xaxes(tickvals=list(filter(lambda x: x < (df_unquantized_acc[eval_label.get(lang, lang)].iloc[0] + 0.20), [0.0, 0.20, 0.40, 0.60, 0.80, 1.0])), row=1 + [base_model_label[x] for x in base_models].index(base_model), col=1 + eval_languages.index(lang))
  # big_fig.update_layout(xaxis=dict(range=[-0.5, len(bits) - 0.5]))
  big_fig.update_xaxes(range=[-0.5, len(bits) - 0.5])
  big_fig.layout.height = 100 + 160 * len(base_models) + (20 if not write_pdf else 35) # base_model
  big_fig.layout.width = 50 + 320 * len(eval_languages) # eval languages

  big_fig.update_xaxes(title_font=dict(size=20))
  big_fig.update_yaxes(title_font=dict(size=20))
  big_fig.update_xaxes(tickfont=dict(size=24))
  big_fig.update_yaxes(tickfont=dict(size=24))
  big_fig.update_layout(
      legend_font_size=20,       # Font size for legend items
      # legend_title_font_size=18  # Font size for legend title
  )

  # big_fig.update_layout(showlegend=False)

  big_fig.update_layout(barmode='overlay')
  big_fig.update_layout(legend=dict(
      orientation="h",
      yanchor="bottom",
      y=1.05,
      xanchor="center",
      x=0.5
  ))

  # just for display purpose, create traces so that legend contains colors.  does not connect with
  # bars
  # print(big_fig.data)
  legend_colors = [
      ("Direct", big_fig.data[1].x[0], big_fig.data[1].y[0], px.colors.qualitative.Plotly[0]),
      ("Cross", big_fig.data[1].x[0], big_fig.data[1].y[0], px.colors.qualitative.Plotly[4]),
      ("Calibration-Free", big_fig.data[1].x[0], big_fig.data[1].y[0], px.colors.qualitative.Plotly[2]),
  ]
  legend_dashes_score = [("Unquantized Score", "/", "solid"), ("Random Guess Score", "-", "dash")]
  legend_dashes = [("G=GPTQ", "/", "solid"), ("S=SliM-LLM", "-", "dash"), ("T=TaCQ", "\\", "longdash")]
  legend_zscores = [("1 Z-Score", "star", "purple"), ("0.5 Z-Score", "star", "green")]
  big_fig.update_traces(showlegend=False).add_traces(
      [
          go.Bar(name=name, x=[x], y=[y], marker_color=color, showlegend=True, zorder=-1, width=[0.2])
          for name, x, y, color in legend_colors
      ]
  )
  big_fig.add_traces(
      [
          go.Scatter(name=name, x=[big_fig.data[1].x[0]], y=[big_fig.data[1].y[0]-0.1], mode='lines', line_dash=dash, line_width=4, marker_color='black', showlegend=True, zorder=-1)
          for name, symbol, dash in legend_dashes_score
      ]
  )
  big_fig.add_traces(
      [
          go.Scatter(name=name, x=[big_fig.data[1].x[0]], y=[big_fig.data[1].y[0]-0.1], mode='none', marker_color='white', showlegend=True, zorder=-1)
          for name, symbol, dash in legend_dashes
      ]
  )
  # big_fig.add_traces(
  #     [
  #         go.Scatter(name=name, x=[big_fig.data[0].x[0]], y=[big_fig.data[0].y[0]], mode='markers', marker_symbol=symbol, marker_color=color, showlegend=True, zorder=-1)
  #         for name, symbol, color in legend_zscores
  #     ]
  # )

  for i, base_model in enumerate(base_models):
    big_fig.update_yaxes(title_text=base_model_label[base_model], row=i+1, col=1)
  big_fig.show()
  if write_pdf:
    big_fig.write_image(write_pdf)

### MultiBlimp Bar

In [ ]:
# eval_languages = ["eng_Latn", "tam_Taml", "ind_Latn", "swh_Latn", "zho_Hans"]
eval_languages = ["eng", "tam", "zhoblimp"]

# Initialize the W&B API
api = Api()

quantization_technique = "ALL"
# bit = 8
num_samples = 128
eval_dataset = "MultiBlimp"
# base_model = "Qwen/Qwen3-8B"
base_model = "All"

# for base_model in ["Qwen/Qwen3-8B", "meta-llama/Llama-3.1-8B-Instruct", "CohereLabs/aya-expanse-8b"]:
for bit in [2]:
  runs = api.runs(path=f"{ENTITY}/{PROJECT}",
                  filters={"$or": [
                          { "$and": [
                            # {"config.quantization_technique": quantization_technique.lower()},
                            # {"config.bit_width": f"{bit}-bit"},
                            {"config.num_calibration_samples": num_samples},
                            {"$or": [
                                {"$and": [
                                  {"config.evaluation_dataset": f"{eval_dataset.lower()}"},
                                  {"created_at": {"$lt": "2026-07-01"}},
                                ]},
                                {"$and": [
                                  {"$or": [
                                    {"config.evaluation_dataset": f"{eval_dataset.lower()}-1000"},
                                    {"config.evaluation_dataset": f"{eval_dataset.lower()}-2000"},
                                  ]},
                                  {"created_at": {"$gt": "2026-07-01"}},
                                ]},
                            ]},
                            # {"config.base_model": base_model},
                            {"config.enable_thinking": False},
                            {"config.output_type": "multiple_choice"},
                            {"$not": {"config.calibration_language": "WuuChinese"}},
                            {"$not": {"config.base_model": "Qwen/Qwen3-14B"}},
                            {"$not": {"$and": [{"created_at": {"$lt": "2026-03-01"}}, {"config.quantization_technique": "gptq"}]}},
                            {"$not": {"config.calibration_language": "Swahili"}},
                            {"$not": {"config.calibration_language": "Indonesian"}},
                          ]
                          },
                          { "$and": [
                            {"config.quantization_technique": "sinq"},
                            # {"config.bit_width": f"{bit}-bit"},
                            {"config.evaluation_dataset": eval_dataset.lower()},
                            # {"config.base_model": base_model},
                            {"config.enable_thinking": False},
                            {"$not": {"config.base_model": "Qwen/Qwen3-14B"}},
                            {"config.output_type": "multiple_choice"},
                            {"created_at": {"$gt": "2026-07-01"}},
                          ]
                          },
                          { "$and" : [
                              {"config.quantization_technique": "Unquantized"},
                              {"config.evaluation_dataset": eval_dataset.lower()},
                              # {"config.base_model": base_model},
                            {"$not": {"config.base_model": "Qwen/Qwen3-14B"}},
                              {"config.enable_thinking": False},
                              {"created_at": {"$lt": "2026-07-01"}},
                          ]}
                      ]}
                  )

  visualize_deliver_bar(eval_languages=eval_languages, runs=runs, base_models=["Qwen/Qwen3-8B", "CohereLabs/aya-expanse-8b", "meta-llama/Llama-3.1-8B-Instruct"], quantization_technique=quantization_technique, bits=["2-bit", "3-bit", "4-bit"], eval_dataset=eval_dataset, custom_title="", display_mean=False, compile=False, vertical_spacing=0.06, horizontal_spacing=0.04, random_guess_score=0.5, write_pdf="multiblimp_barplot_reduced_errorbar_final.pdf")

Pre-AGG


,Base Model,Quantization Technique,Bit Width,Calibration Language,eng,tam,zhoblimp
0,Qwen/Qwen3-8B,gptq,2-bit,English,0.681818,0.620419,0.562034
27,Qwen/Qwen3-8B,gptq,2-bit,English,0.637662,0.685864,0.561045
28,Qwen/Qwen3-8B,gptq,2-bit,English,0.715584,0.643979,0.557853
3,Qwen/Qwen3-8B,gptq,3-bit,English,0.977922,0.814136,0.804322
33,Qwen/Qwen3-8B,gptq,3-bit,English,0.972727,0.782723,0.805085
34,Qwen/Qwen3-8B,gptq,3-bit,English,0.966234,0.732984,0.815650
6,Qwen/Qwen3-8B,gptq,4-bit,English,0.997403,0.895288,1.000000
1,Qwen/Qwen3-8B,gptq,2-bit,Tamil,0.624675,0.706806,0.567147
29,Qwen/Qwen3-8B,gptq,2-bit,Tamil,0.616883,0.787958,0.583927
30,Qwen/Qwen3-8B,gptq,2-bit,Tamil,0.589610,0.793194,0.566723


,Base Model,Quantization Technique,Bit Width,Calibration Language,eng,tam,zhoblimp
0,Qwen/Qwen3-8B,slimllm,2-bit,English,0.937662,0.479058,0.572825
24,Qwen/Qwen3-8B,slimllm,2-bit,English,0.923377,0.672775,0.579040
25,Qwen/Qwen3-8B,slimllm,2-bit,English,0.928571,0.696335,0.588277
4,Qwen/Qwen3-8B,slimllm,3-bit,English,0.970130,0.801047,0.791017
30,Qwen/Qwen3-8B,slimllm,3-bit,English,0.967532,0.714660,0.786723
31,Qwen/Qwen3-8B,slimllm,3-bit,English,0.958442,0.869110,0.803701
16,Qwen/Qwen3-8B,slimllm,4-bit,English,0.967532,0.908377,0.814802
1,Qwen/Qwen3-8B,slimllm,2-bit,Tamil,0.723377,0.814136,0.629153
26,Qwen/Qwen3-8B,slimllm,2-bit,Tamil,0.688312,0.848168,0.577740
27,Qwen/Qwen3-8B,slimllm,2-bit,Tamil,0.705195,0.790576,0.596384


,Base Model,Quantization Technique,Bit Width,Calibration Language,eng,tam,zhoblimp
4,Qwen/Qwen3-8B,tacq,2-bit,English,0.962338,0.675393,0.764576
27,Qwen/Qwen3-8B,tacq,2-bit,English,0.979221,0.489529,0.777542
28,Qwen/Qwen3-8B,tacq,2-bit,English,0.968831,0.552356,0.786554
0,Qwen/Qwen3-8B,tacq,3-bit,English,0.971429,0.816754,0.817599
33,Qwen/Qwen3-8B,tacq,3-bit,English,0.968831,0.835079,0.809266
34,Qwen/Qwen3-8B,tacq,3-bit,English,0.970130,0.808901,0.809322
18,Qwen/Qwen3-8B,tacq,4-bit,English,0.970130,0.934555,0.810791
6,Qwen/Qwen3-8B,tacq,2-bit,Tamil,0.958442,0.869110,0.792062
29,Qwen/Qwen3-8B,tacq,2-bit,Tamil,0.962338,0.871728,0.804633
30,Qwen/Qwen3-8B,tacq,2-bit,Tamil,0.953247,0.861257,0.807034


Contrasting Acc and Stderr
GPTQ


...

,0
eng,2.307359
tam,2.059337
zhoblimp,1.624708


[[0, 'EN'], [1, 'TA'], [4, 'ZH']]
where TA located
['EN' 'TA' 'ZH']
mean_ppl


,0
eng,2.696104
tam,2.094241
zhoblimp,2.111186


[[0, 'EN'], [1, 'TA'], [4, 'ZH']]
where TA located
['EN' 'TA' 'ZH']
mean_ppl


,0
eng,2.941558
tam,2.709424
zhoblimp,2.378418


In [ ]:
eval_languages = ["eng", "tam", "idn", "swa", "zhoblimp"]
# eval_languages = ["eng", "tam", "idn"]

# Initialize the W&B API
api = Api()

quantization_technique = "ALL"
# bit = 8
num_samples = 128
eval_dataset = "MultiBlimp"
# base_model = "Qwen/Qwen3-8B"
base_model = "All"

# for base_model in ["Qwen/Qwen3-8B", "meta-llama/Llama-3.1-8B-Instruct", "CohereLabs/aya-expanse-8b"]:
for bit in [2]:
  runs = api.runs(path=f"{ENTITY}/{PROJECT}",
                  filters={"$or": [
                          { "$and": [
                            # {"config.quantization_technique": quantization_technique.lower()},
                            # {"config.bit_width": f"{bit}-bit"},
                            {"config.num_calibration_samples": num_samples},
                            {"config.evaluation_dataset": f"{eval_dataset.lower()}"},
                              {"config.base_model": "Qwen/Qwen3-14B"},
                            # {"config.base_model": base_model},
                            {"config.enable_thinking": False},
                            {"config.output_type": "multiple_choice"},
                            {"$not": {"config.calibration_language": "WuuChinese"}},
                            {"$not": {"$and": [{"created_at": {"$lt": "2026-03-01"}}, {"config.quantization_technique": "gptq"}]}},
                            {"created_at": {"$gt": "2026-07-01"}},
                            # {"$not": {"config.calibration_language": "Swahili"}},
                            # {"$not": {"config.calibration_language": "Chinese"}},
                          ]
                          },
                          { "$and": [
                            {"config.quantization_technique": "sinq"},
                            # {"config.bit_width": f"{bit}-bit"},
                            {"config.evaluation_dataset": eval_dataset.lower()},
                            # {"config.base_model": base_model},
                            {"config.enable_thinking": False},
                              {"config.base_model": "Qwen/Qwen3-14B"},
                            {"config.output_type": "multiple_choice"},
                            {"created_at": {"$gt": "2026-07-01"}},
                          ]
                          },
                          { "$and" : [
                              {"config.quantization_technique": "Unquantized"},
                              {"config.evaluation_dataset": eval_dataset.lower()},
                              # {"config.base_model": base_model},
                              {"config.enable_thinking": False},
                              {"config.base_model": "Qwen/Qwen3-14B"},
                              {"created_at": {"$gt": "2026-07-01"}},
                          ]}
                      ]}
                  )

  visualize_deliver_bar(eval_languages=eval_languages, runs=runs, base_models=["Qwen/Qwen3-14B"], quantization_technique=quantization_technique, bits=["2-bit", "3-bit"], eval_dataset=eval_dataset, custom_title="", display_mean=False, compile=False, vertical_spacing=0.038, horizontal_spacing=0.040, random_guess_score=0.5, write_pdf="multiblimp_14b_barplot_reduced_errorbar_final.pdf")

Pre-AGG


,Base Model,Quantization Technique,Bit Width,Calibration Language,eng,tam,idn,swa,zhoblimp
0,Qwen/Qwen3-14B,gptq,2-bit,English,0.789610,0.672775,0.528947,0.512125,0.571328
3,Qwen/Qwen3-14B,gptq,3-bit,English,0.972727,0.866492,0.628947,0.461125,0.811356
2,Qwen/Qwen3-14B,gptq,2-bit,Tamil,0.661039,0.856021,0.494737,0.492625,0.577147
5,Qwen/Qwen3-14B,gptq,3-bit,Tamil,0.959740,0.910995,0.626316,0.471250,0.755763
1,Qwen/Qwen3-14B,gptq,2-bit,Indonesian,0.689610,0.523560,0.518421,0.481250,0.586186
4,Qwen/Qwen3-14B,gptq,3-bit,Indonesian,0.970130,0.905759,0.607895,0.457500,0.759124
8,Qwen/Qwen3-14B,gptq,2-bit,Swahili,0.618182,0.612565,0.521053,0.473375,0.579435
6,Qwen/Qwen3-14B,gptq,3-bit,Swahili,0.967532,0.884817,0.610526,0.465750,0.769181
9,Qwen/Qwen3-14B,gptq,2-bit,Chinese,0.693506,0.651832,0.555263,0.498875,0.611469
7,Qwen/Qwen3-14B,gptq,3-bit,Chinese,0.966234,0.882199,0.600000,0.478750,0.802062


,Base Model,Quantization Technique,Bit Width,Calibration Language,eng,tam,idn,swa,zhoblimp


,Base Model,Quantization Technique,Bit Width,Calibration Language,eng,tam,idn,swa,zhoblimp
0,Qwen/Qwen3-14B,tacq,2-bit,English,0.980519,0.678010,0.505263,0.476500,0.793616
3,Qwen/Qwen3-14B,tacq,3-bit,English,0.963636,0.895288,0.607895,0.441875,0.768079
2,Qwen/Qwen3-14B,tacq,2-bit,Tamil,0.958442,0.910995,0.526316,0.456625,0.825734
5,Qwen/Qwen3-14B,tacq,3-bit,Tamil,0.972727,0.918848,0.623684,0.474125,0.781554
1,Qwen/Qwen3-14B,tacq,2-bit,Indonesian,0.961039,0.636126,0.621053,0.463250,0.834633
4,Qwen/Qwen3-14B,tacq,3-bit,Indonesian,0.955844,0.897906,0.607895,0.476375,0.769661
6,Qwen/Qwen3-14B,tacq,2-bit,Swahili,0.967532,0.738220,0.550000,0.443875,0.818192
8,Qwen/Qwen3-14B,tacq,3-bit,Swahili,0.964935,0.910995,0.626316,0.471625,0.784605
7,Qwen/Qwen3-14B,tacq,2-bit,Chinese,0.962338,0.544503,0.442105,0.477375,0.855395
9,Qwen/Qwen3-14B,tacq,3-bit,Chinese,0.968831,0.903141,0.615789,0.470375,0.790876


Contrasting Acc and Stderr
GPTQ


...

,Base Model,Quantization Technique,Calibration Language,Bit Width,eng,tam,idn,swa,zhoblimp


TACQ


,Base Model,Quantization Technique,Calibration Language,Bit Width,eng,tam,idn,swa,zhoblimp
2,Qwen/Qwen3-14B,tacq,English,2-bit,0.980519,0.678010,0.505263,0.476500,0.793616
3,Qwen/Qwen3-14B,tacq,English,3-bit,0.963636,0.895288,0.607895,0.441875,0.768079
8,Qwen/Qwen3-14B,tacq,Tamil,2-bit,0.958442,0.910995,0.526316,0.456625,0.825734
9,Qwen/Qwen3-14B,tacq,Tamil,3-bit,0.972727,0.918848,0.623684,0.474125,0.781554
4,Qwen/Qwen3-14B,tacq,Indonesian,2-bit,0.961039,0.636126,0.621053,0.463250,0.834633
5,Qwen/Qwen3-14B,tacq,Indonesian,3-bit,0.955844,0.897906,0.607895,0.476375,0.769661
6,Qwen/Qwen3-14B,tacq,Swahili,2-bit,0.967532,0.738220,0.550000,0.443875,0.818192
7,Qwen/Qwen3-14B,tacq,Swahili,3-bit,0.964935,0.910995,0.626316,0.471625,0.784605
0,Qwen/Qwen3-14B,tacq,Chinese,2-bit,0.962338,0.544503,0.442105,0.477375,0.855395
1,Qwen/Qwen3-14B,tacq,Chinese,3-bit,0.968831,0.903141,0.615789,0.470375,0.790876


,Base Model,Quantization Technique,Calibration Language,Bit Width,eng,tam,idn,swa,zhoblimp


[[0, 'EN'], [1, 'TA'], [2, 'ID'], [3, 'SW'], [4, 'ZH']]
where TA located
[]
[[0, 'EN'], [1, 'TA'], [2, 'ID'], [3, 'SW'], [4, 'ZH']]
where TA located
[]


### Belebele Bar

In [ ]:
# eval_languages = ["eng_Latn", "tam_Taml", "ind_Latn", "swh_Latn", "zho_Hans"]
eval_languages = ["eng_Latn", "tam_Taml", "ind_Latn", "swh_Latn", "zho_Hans"]

# Initialize the W&B API
api = Api()

quantization_technique = "ALL"
# bit = 8
num_samples = 128
eval_dataset = "Belebele"
# base_model = "Qwen/Qwen3-8B"
base_model = "All"

# for base_model in ["Qwen/Qwen3-8B", "meta-llama/Llama-3.1-8B-Instruct", "CohereLabs/aya-expanse-8b"]:
for bit in [2]:
  runs = api.runs(path=f"{ENTITY}/{PROJECT}",
                  filters={"$or": [
                          { "$and": [
                            # {"config.quantization_technique": quantization_technique.lower()},
                            # {"config.bit_width": f"{bit}-bit"},
                            {"config.num_calibration_samples": num_samples},
                            {"$or": [
                                {"$and": [
                                  {"config.evaluation_dataset": f"{eval_dataset.lower()}"},
                                  {"created_at": {"$lt": "2026-07-01"}},
                                ]},
                                {"$and": [
                                  {"$or": [
                                    {"config.evaluation_dataset": f"{eval_dataset.lower()}-1000"},
                                    {"config.evaluation_dataset": f"{eval_dataset.lower()}-2000"},
                                  ]},
                                  {"created_at": {"$gt": "2026-07-01"}},
                                ]},
                            ]},
                            # {"config.base_model": base_model},
                            {"config.enable_thinking": False},
                            {"config.output_type": "multiple_choice"},
                            {"$not": {"config.calibration_language": "WuuChinese"}},
                            {"$not": {"$and": [{"created_at": {"$lt": "2026-03-01"}}, {"config.quantization_technique": "gptq"}]}},
                          ]
                          },
                          { "$and": [
                            {"config.quantization_technique": "sinq"},
                            # {"config.bit_width": f"{bit}-bit"},
                            {"config.evaluation_dataset": eval_dataset.lower()},
                            # {"config.base_model": base_model},
                            {"config.enable_thinking": False},
                            {"config.output_type": "multiple_choice"},
                            {"created_at": {"$gt": "2026-07-01"}},
                          ]
                          },
                          { "$and" : [
                              {"config.quantization_technique": "Unquantized"},
                              {"config.evaluation_dataset": eval_dataset.lower()},
                              # {"config.base_model": base_model},
                              {"config.enable_thinking": False},
                              {"created_at": {"$lt": "2026-07-01"}},
                          ]}
                      ]}
                  )

  # visualize_delta_unquantized_globalmmlulite(eval_languages=eval_languages, runs=runs, base_model=base_model, quantization_technique=quantization_technique, bit_width=bit, eval_dataset=eval_dataset, custom_title="")
  # visualize_delta_unquantized_deliver_bar(eval_languages=eval_languages, runs=runs, base_model=base_model, quantization_technique=quantization_technique, bits=["2-bit", "3-bit", "4-bit"], eval_dataset=eval_dataset, custom_title="", display_mean=False, compile=False)
  visualize_deliver_bar(eval_languages=eval_languages, runs=runs, base_models=["Qwen/Qwen3-8B", "CohereLabs/aya-expanse-8b", "meta-llama/Llama-3.1-8B-Instruct"], quantization_technique=quantization_technique, bits=["2-bit", "3-bit", "4-bit"], eval_dataset=eval_dataset, custom_title="", display_mean=False, compile=False, vertical_spacing=0.05, horizontal_spacing=0.025, random_guess_score=0.25, write_pdf="belebele_barplot_errorbar_final.pdf")

Pre-AGG


,Base Model,Quantization Technique,Bit Width,Calibration Language,eng_Latn,tam_Taml,ind_Latn,swh_Latn,zho_Hans
0,Qwen/Qwen3-8B,gptq,2-bit,English,0.223333,0.245556,0.257778,0.240000,0.233333
45,Qwen/Qwen3-8B,gptq,2-bit,English,0.215556,0.244444,0.245556,0.245556,0.246667
46,Qwen/Qwen3-8B,gptq,2-bit,English,0.253333,0.235556,0.254444,0.252222,0.264444
5,Qwen/Qwen3-8B,gptq,3-bit,English,0.856667,0.448889,0.728889,0.336667,0.814444
55,Qwen/Qwen3-8B,gptq,3-bit,English,0.863333,0.465556,0.771111,0.365556,0.837778
...,...,...,...,...,...,...,...,...,...
19,meta-llama/Llama-3.1-8B-Instruct,gptq,2-bit,Chinese,0.240000,0.260000,0.244444,0.255556,0.266667
73,meta-llama/Llama-3.1-8B-Instruct,gptq,2-bit,Chinese,0.246667,0.260000,0.272222,0.237778,0.247778
74,meta-llama/Llama-3.1-8B-Instruct,gptq,2-bit,Chinese,0.241111,0.234444,0.222222,0.256667,0.232222
24,meta-llama/Llama-3.1-8B-Instruct,gptq,3-bit,Chinese,0.735556,0.225556,0.461111,0.284444,0.750000


,Base Model,Quantization Technique,Bit Width,Calibration Language,eng_Latn,tam_Taml,ind_Latn,swh_Latn,zho_Hans
15,Qwen/Qwen3-8B,slimllm,2-bit,English,0.230000,0.233333,0.228889,0.230000,0.228889
42,Qwen/Qwen3-8B,slimllm,2-bit,English,0.228889,0.228889,0.228889,0.226667,0.228889
43,Qwen/Qwen3-8B,slimllm,2-bit,English,0.226667,0.228889,0.230000,0.228889,0.227778
20,Qwen/Qwen3-8B,slimllm,3-bit,English,0.851111,0.410000,0.731111,0.335556,0.831111
52,Qwen/Qwen3-8B,slimllm,3-bit,English,0.870000,0.437778,0.754444,0.338889,0.836667
...,...,...,...,...,...,...,...,...,...
4,meta-llama/Llama-3.1-8B-Instruct,slimllm,2-bit,Chinese,0.226667,0.250000,0.273333,0.241111,0.227778
70,meta-llama/Llama-3.1-8B-Instruct,slimllm,2-bit,Chinese,0.221111,0.231111,0.224444,0.243333,0.225556
71,meta-llama/Llama-3.1-8B-Instruct,slimllm,2-bit,Chinese,0.222222,0.263333,0.264444,0.230000,0.251111
9,meta-llama/Llama-3.1-8B-Instruct,slimllm,3-bit,Chinese,0.681111,0.296667,0.491111,0.287778,0.671111


,Base Model,Quantization Technique,Bit Width,Calibration Language,eng_Latn,tam_Taml,ind_Latn,swh_Latn,zho_Hans
15,Qwen/Qwen3-8B,tacq,2-bit,English,0.587778,0.230000,0.347778,0.230000,0.540000
45,Qwen/Qwen3-8B,tacq,2-bit,English,0.577778,0.268889,0.352222,0.275556,0.503333
46,Qwen/Qwen3-8B,tacq,2-bit,English,0.626667,0.233333,0.351111,0.244444,0.530000
20,Qwen/Qwen3-8B,tacq,3-bit,English,0.904444,0.563333,0.831111,0.421111,0.867778
55,Qwen/Qwen3-8B,tacq,3-bit,English,0.911111,0.564444,0.807778,0.418889,0.862222
...,...,...,...,...,...,...,...,...,...
4,meta-llama/Llama-3.1-8B-Instruct,tacq,2-bit,Chinese,0.235556,0.228889,0.231111,0.228889,0.247778
73,meta-llama/Llama-3.1-8B-Instruct,tacq,2-bit,Chinese,0.302222,0.228889,0.227778,0.233333,0.342222
74,meta-llama/Llama-3.1-8B-Instruct,tacq,2-bit,Chinese,0.263333,0.223333,0.235556,0.257778,0.260000
9,meta-llama/Llama-3.1-8B-Instruct,tacq,3-bit,Chinese,0.804444,0.435556,0.675556,0.391111,0.766667


Contrasting Acc and Stderr
GPTQ


...

,0
eng_Latn,1.197407
tam_Taml,1.251481
ind_Latn,1.235185
swh_Latn,1.234074
zho_Hans,1.221111


[[0, 'EN'], [1, 'TA'], [2, 'ID'], [3, 'SW'], [4, 'ZH']]
where TA located
['EN' 'TA' 'ID' 'SW' 'ZH']
mean_ppl


,0
eng_Latn,2.703333
tam_Taml,1.268889
ind_Latn,1.822222
swh_Latn,1.351111
zho_Hans,2.090000


[[0, 'EN'], [1, 'TA'], [2, 'ID'], [3, 'SW'], [4, 'ZH']]
where TA located
['EN' 'TA' 'ID' 'SW' 'ZH']
mean_ppl


,0
eng_Latn,4.290000
tam_Taml,2.361111
ind_Latn,3.544444
swh_Latn,2.422222
zho_Hans,3.984444


In [ ]:
eval_languages = ["eng_Latn", "tam_Taml", "ind_Latn", "swh_Latn", "zho_Hans"]
# eval_languages = ["eng_Latn", "tam_Taml", "ind_Latn"]

# Initialize the W&B API
api = Api()

quantization_technique = "ALL"
# bit = 8
num_samples = 128
eval_dataset = "Belebele"
# base_model = "Qwen/Qwen3-8B"
base_model = "All"

# for base_model in ["Qwen/Qwen3-8B", "meta-llama/Llama-3.1-8B-Instruct", "CohereLabs/aya-expanse-8b"]:
for bit in [2]:
  runs = api.runs(path=f"{ENTITY}/{PROJECT}",
                  filters={"$or": [
                          { "$and": [
                            # {"config.quantization_technique": quantization_technique.lower()},
                            # {"config.bit_width": f"{bit}-bit"},
                            { "$or": [
                              {"config.evaluation_dataset": eval_dataset.lower()},
                              {"config.evaluation_dataset": f"{eval_dataset.lower()}-1000"},
                              {"config.evaluation_dataset": f"{eval_dataset.lower()}-2000"},
                            ]},
                            # {"config.base_model": base_model},
                              {"config.base_model": "Qwen/Qwen3-14B"},
                            {"created_at": {"$gt": "2026-07-01"}},
                          ]
                          },
                          { "$and": [
                            {"config.quantization_technique": "sinq"},
                            # {"config.bit_width": f"{bit}-bit"},
                            {"config.evaluation_dataset": eval_dataset.lower()},
                            # {"config.base_model": base_model},
                            {"config.enable_thinking": False},
                              {"config.base_model": "Qwen/Qwen3-14B"},
                            {"config.output_type": "multiple_choice"},
                            {"created_at": {"$gt": "2026-07-01"}},
                          ]
                          },
                          { "$and" : [
                              {"config.quantization_technique": "Unquantized"},
                              {"config.evaluation_dataset": eval_dataset.lower()},
                              # {"config.base_model": base_model},
                              {"config.base_model": "Qwen/Qwen3-14B"},
                              {"config.enable_thinking": False},
                              {"created_at": {"$gt": "2026-07-01"}},
                          ]}
                      ]}
                  )
  # visualize_delta_unquantized_globalmmlulite(eval_languages=eval_languages, runs=runs, base_model=base_model, quantization_technique=quantization_technique, bit_width=bit, eval_dataset=eval_dataset, custom_title="")
  # visualize_delta_unquantized_deliver_bar(eval_languages=eval_languages, runs=runs, base_model=base_model, quantization_technique=quantization_technique, bits=["2-bit", "3-bit", "4-bit"], eval_dataset=eval_dataset, custom_title="", display_mean=False, compile=False)
  visualize_deliver_bar(eval_languages=eval_languages, runs=runs, base_models=["Qwen/Qwen3-14B"], quantization_technique=quantization_technique, bits=["2-bit", "3-bit"], eval_dataset=eval_dataset, custom_title="", display_mean=False, compile=False, vertical_spacing=0.038, horizontal_spacing=0.04, random_guess_score=0.25, write_pdf="belebele_14b_barplot_errorbar_final.pdf")

Pre-AGG


,Base Model,Quantization Technique,Bit Width,Calibration Language,eng_Latn,tam_Taml,ind_Latn,swh_Latn,zho_Hans
7,Qwen/Qwen3-14B,gptq,2-bit,English,0.235556,0.240000,0.240000,0.230000,0.222222
2,Qwen/Qwen3-14B,gptq,3-bit,English,0.915556,0.608889,0.853333,0.471111,0.878889
1,Qwen/Qwen3-14B,gptq,2-bit,Tamil,0.242222,0.243333,0.248889,0.224444,0.245556
4,Qwen/Qwen3-14B,gptq,3-bit,Tamil,0.894444,0.735556,0.853333,0.490000,0.877778
0,Qwen/Qwen3-14B,gptq,2-bit,Indonesian,0.250000,0.242222,0.238889,0.230000,0.241111
3,Qwen/Qwen3-14B,gptq,3-bit,Indonesian,0.903333,0.657778,0.844444,0.461111,0.880000
9,Qwen/Qwen3-14B,gptq,2-bit,Swahili,0.223333,0.244444,0.232222,0.225556,0.243333
5,Qwen/Qwen3-14B,gptq,3-bit,Swahili,0.903333,0.643333,0.842222,0.548889,0.866667
8,Qwen/Qwen3-14B,gptq,2-bit,Chinese,0.225556,0.231111,0.227778,0.235556,0.241111
6,Qwen/Qwen3-14B,gptq,3-bit,Chinese,0.904444,0.652222,0.831111,0.447778,0.884444


,Base Model,Quantization Technique,Bit Width,Calibration Language,eng_Latn,tam_Taml,ind_Latn,swh_Latn,zho_Hans


,Base Model,Quantization Technique,Bit Width,Calibration Language,eng_Latn,tam_Taml,ind_Latn,swh_Latn,zho_Hans
0,Qwen/Qwen3-14B,tacq,2-bit,English,0.823333,0.244444,0.435556,0.264444,0.692222
3,Qwen/Qwen3-14B,tacq,3-bit,English,0.925556,0.674444,0.865556,0.513333,0.898889
2,Qwen/Qwen3-14B,tacq,2-bit,Tamil,0.765556,0.600000,0.608889,0.288889,0.712222
5,Qwen/Qwen3-14B,tacq,3-bit,Tamil,0.924444,0.750000,0.871111,0.535556,0.906667
1,Qwen/Qwen3-14B,tacq,2-bit,Indonesian,0.774444,0.295556,0.737778,0.304444,0.744444
4,Qwen/Qwen3-14B,tacq,3-bit,Indonesian,0.934444,0.703333,0.876667,0.571111,0.896667
6,Qwen/Qwen3-14B,tacq,2-bit,Swahili,0.668889,0.271111,0.550000,0.408889,0.613333
8,Qwen/Qwen3-14B,tacq,3-bit,Swahili,0.926667,0.720000,0.870000,0.630000,0.888889
7,Qwen/Qwen3-14B,tacq,2-bit,Chinese,0.810000,0.280000,0.477778,0.290000,0.775556
9,Qwen/Qwen3-14B,tacq,3-bit,Chinese,0.930000,0.681111,0.880000,0.497778,0.895556


Contrasting Acc and Stderr
GPTQ


...

,Base Model,Quantization Technique,Calibration Language,Bit Width,eng_Latn,tam_Taml,ind_Latn,swh_Latn,zho_Hans


TACQ


,Base Model,Quantization Technique,Calibration Language,Bit Width,eng_Latn,tam_Taml,ind_Latn,swh_Latn,zho_Hans
2,Qwen/Qwen3-14B,tacq,English,2-bit,0.823333,0.244444,0.435556,0.264444,0.692222
3,Qwen/Qwen3-14B,tacq,English,3-bit,0.925556,0.674444,0.865556,0.513333,0.898889
8,Qwen/Qwen3-14B,tacq,Tamil,2-bit,0.765556,0.600000,0.608889,0.288889,0.712222
9,Qwen/Qwen3-14B,tacq,Tamil,3-bit,0.924444,0.750000,0.871111,0.535556,0.906667
4,Qwen/Qwen3-14B,tacq,Indonesian,2-bit,0.774444,0.295556,0.737778,0.304444,0.744444
5,Qwen/Qwen3-14B,tacq,Indonesian,3-bit,0.934444,0.703333,0.876667,0.571111,0.896667
6,Qwen/Qwen3-14B,tacq,Swahili,2-bit,0.668889,0.271111,0.550000,0.408889,0.613333
7,Qwen/Qwen3-14B,tacq,Swahili,3-bit,0.926667,0.720000,0.870000,0.630000,0.888889
0,Qwen/Qwen3-14B,tacq,Chinese,2-bit,0.810000,0.280000,0.477778,0.290000,0.775556
1,Qwen/Qwen3-14B,tacq,Chinese,3-bit,0.930000,0.681111,0.880000,0.497778,0.895556


,Base Model,Quantization Technique,Calibration Language,Bit Width,eng_Latn,tam_Taml,ind_Latn,swh_Latn,zho_Hans


[[0, 'EN'], [1, 'TA'], [2, 'ID'], [3, 'SW'], [4, 'ZH']]
where TA located
[]
[[0, 'EN'], [1, 'TA'], [2, 'ID'], [3, 'SW'], [4, 'ZH']]
where TA located
[]


### Massive Scenario Bar

In [ ]:
eval_languages = ['en-US', 'id-ID', 'ta-IN', 'sw-KE', 'zh-CN']

# Initialize the W&B API
api = Api()

quantization_technique = "GPTQ"
# bit = 8
num_samples = 128
eval_dataset = "massive"
eval_dataset_variant = "-scenario"
# base_model = "Qwen/Qwen3-8B"
base_model = "All"

# for base_model in ["Qwen/Qwen3-8B", "meta-llama/Llama-3.1-8B-Instruct", "CohereLabs/aya-expanse-8b"]:
for bit in [3]:
  runs = api.runs(path=f"{ENTITY}/{PROJECT}",
                  filters={"$or": [
                          { "$and": [
                            # {"config.quantization_technique": quantization_technique.lower()},
                            # {"config.bit_width": f"{bit}-bit"},
                            {"config.num_calibration_samples": num_samples},
                            # {"config.evaluation_dataset": eval_dataset.lower()},
                            # {"config.evaluation_dataset": f"{eval_dataset}{eval_dataset_variant}"},
                            {"$or": [
                                {"$and": [
                                  {"config.evaluation_dataset": f"{eval_dataset.lower()}{eval_dataset_variant}"},
                                  {"created_at": {"$lt": "2026-07-01"}},
                                ]},
                                {"$and": [
                                  {"$or": [
                                    {"config.evaluation_dataset": f"{eval_dataset.lower()}{eval_dataset_variant}-1000"},
                                    {"config.evaluation_dataset": f"{eval_dataset.lower()}{eval_dataset_variant}-2000"},
                                  ]},
                                  {"created_at": {"$gt": "2026-07-01"}},
                                ]},
                            ]},
                            # {"config.base_model": base_model},
                            {"config.enable_thinking": False},
                            {"config.output_type": "multiple_choice"},
                            {"$not": {"config.calibration_language": "WuuChinese"}},
                            # {"$not": {"config.calibration_language": "Chinese"}},
                            # {"$not": {"config.calibration_language": "Swahili"}},
                            # {"$not": {"config.calibration_language": "Indonesian"}},
                            # {"created_at": {"$lt": "2026-07-01"}},
                          ]
                          },
                          { "$and": [
                            {"config.quantization_technique": "sinq"},
                            # {"config.bit_width": f"{bit}-bit"},
                            {"config.evaluation_dataset": f"{eval_dataset}{eval_dataset_variant}"},
                            # {"config.base_model": base_model},
                            {"config.enable_thinking": False},
                            {"config.output_type": "multiple_choice"},
                            {"created_at": {"$gt": "2026-07-01"}},
                          ]
                          },
                          { "$and" : [
                              {"config.quantization_technique": "Unquantized"},
                              # {"config.evaluation_dataset": eval_dataset.lower()},
                            {"config.evaluation_dataset": f"{eval_dataset}{eval_dataset_variant}"},
                              # {"config.base_model": base_model},
                              {"config.enable_thinking": False},
                            {"created_at": {"$lt": "2026-07-01"}},
                          ]}
                      ]}
                  )

  # visualize_delta_unquantized_deliver_bar(eval_languages=eval_languages, runs=runs, base_model=base_model, quantization_technique=quantization_technique, bits=['2-bit', '3-bit'], eval_dataset=eval_dataset, custom_title="", display_mean=False, compile=True)
  visualize_deliver_bar(eval_languages=eval_languages, runs=runs, base_models=["Qwen/Qwen3-8B", "CohereLabs/aya-expanse-8b"], quantization_technique=quantization_technique, bits=['2-bit', '3-bit', '4-bit'], eval_dataset=eval_dataset, custom_title="", display_mean=False, compile=True, vertical_spacing=0.08, horizontal_spacing=0.025, random_guess_score=1/18, write_pdf="massive_barplot_sinq_final.pdf")


Pre-AGG


,Base Model,Quantization Technique,Bit Width,Calibration Language,en-US,ta-IN,id-ID,sw-KE,zh-CN
5,Qwen/Qwen3-8B,gptq,2-bit,English,0.075319,0.063887,0.062206,0.065905,0.068931
30,Qwen/Qwen3-8B,gptq,2-bit,English,0.083389,0.061197,0.088097,0.069603,0.066241
31,Qwen/Qwen3-8B,gptq,2-bit,English,0.078682,0.065232,0.081708,0.065232,0.060188
0,Qwen/Qwen3-8B,gptq,3-bit,English,0.474445,0.364492,0.457969,0.259919,0.483524
40,Qwen/Qwen3-8B,gptq,3-bit,English,0.499328,0.408204,0.458305,0.300269,0.496301
41,Qwen/Qwen3-8B,gptq,3-bit,English,0.532616,0.422999,0.522192,0.319435,0.548420
20,Qwen/Qwen3-8B,gptq,4-bit,English,0.486550,0.438803,0.464694,0.363820,0.486886
7,Qwen/Qwen3-8B,gptq,2-bit,Tamil,0.062878,0.062878,0.080363,0.071957,0.068258
34,Qwen/Qwen3-8B,gptq,2-bit,Tamil,0.062206,0.065568,0.057498,0.063551,0.063551
35,Qwen/Qwen3-8B,gptq,2-bit,Tamil,0.057498,0.058507,0.060861,0.064223,0.061533


,Base Model,Quantization Technique,Bit Width,Calibration Language,en-US,ta-IN,id-ID,sw-KE,zh-CN
7,Qwen/Qwen3-8B,slimllm,2-bit,English,0.111298,0.058843,0.083053,0.066913,0.097176
2,Qwen/Qwen3-8B,slimllm,3-bit,English,0.503026,0.383995,0.473100,0.300605,0.497310
27,Qwen/Qwen3-8B,slimllm,3-bit,English,0.483524,0.307666,0.440484,0.271015,0.485878
28,Qwen/Qwen3-8B,slimllm,3-bit,English,0.507061,0.355414,0.465030,0.300605,0.500000
0,Qwen/Qwen3-8B,slimllm,4-bit,English,0.503026,0.451580,0.497983,0.381641,0.525891
9,Qwen/Qwen3-8B,slimllm,2-bit,Tamil,0.105582,0.099866,0.099193,0.093141,0.091459
4,Qwen/Qwen3-8B,slimllm,3-bit,Tamil,0.419637,0.415266,0.386012,0.273033,0.455952
31,Qwen/Qwen3-8B,slimllm,3-bit,Tamil,0.434432,0.393746,0.398790,0.238399,0.438467
32,Qwen/Qwen3-8B,slimllm,3-bit,Tamil,0.418292,0.436113,0.384331,0.259247,0.485878
20,Qwen/Qwen3-8B,slimllm,4-bit,Tamil,0.483524,0.456288,0.469738,0.363820,0.501009


,Base Model,Quantization Technique,Bit Width,Calibration Language,en-US,ta-IN,id-ID,sw-KE,zh-CN
10,Qwen/Qwen3-8B,tacq,2-bit,English,0.421318,0.084734,0.261264,0.106254,0.371217
5,Qwen/Qwen3-8B,tacq,3-bit,English,0.493948,0.421991,0.481843,0.320444,0.507397
30,Qwen/Qwen3-8B,tacq,3-bit,English,0.485205,0.360121,0.442838,0.292199,0.489576
31,Qwen/Qwen3-8B,tacq,3-bit,English,0.481506,0.361130,0.438803,0.287492,0.493948
0,Qwen/Qwen3-8B,tacq,4-bit,English,0.518494,0.463013,0.501009,0.373235,0.515131
12,Qwen/Qwen3-8B,tacq,2-bit,Tamil,0.298252,0.185609,0.207801,0.099529,0.358104
7,Qwen/Qwen3-8B,tacq,3-bit,Tamil,0.497310,0.451244,0.476126,0.344990,0.517821
34,Qwen/Qwen3-8B,tacq,3-bit,Tamil,0.467048,0.458642,0.466039,0.330868,0.501681
35,Qwen/Qwen3-8B,tacq,3-bit,Tamil,0.502690,0.467720,0.490249,0.337592,0.504035
2,Qwen/Qwen3-8B,tacq,4-bit,Tamil,0.506389,0.447545,0.486550,0.371890,0.512105


Contrasting Acc and Stderr
GPTQ


...

,0
en-US,0.840955
ta-IN,0.329523
id-ID,0.667787
sw-KE,0.353396
zh-CN,0.729657


[[0, 'EN'], [1, 'TA'], [2, 'ID'], [3, 'SW'], [4, 'ZH']]
where TA located
['EN' 'ID' 'SW' 'ZH']
mean_ppl


,0
en-US,2.370208
ta-IN,1.583053
id-ID,2.300605
sw-KE,1.207465
zh-CN,2.366174


[[0, 'EN'], [1, 'TA'], [2, 'ID'], [3, 'SW'], [4, 'ZH']]
where TA located
['EN' 'ID' 'SW' 'ZH']
mean_ppl


,0
en-US,2.409549
ta-IN,1.866510
id-ID,2.364492
sw-KE,1.424681
zh-CN,2.412239


In [ ]:
eval_languages = ['en-US', 'id-ID', 'ta-IN', 'sw-KE', 'zh-CN']

# Initialize the W&B API
api = Api()

quantization_technique = "GPTQ"
# bit = 8
num_samples = 128
eval_dataset = "massive"
eval_dataset_variant = "-scenario"
# base_model = "Qwen/Qwen3-8B"
base_model = "All"

# for base_model in ["Qwen/Qwen3-8B", "meta-llama/Llama-3.1-8B-Instruct", "CohereLabs/aya-expanse-8b"]:
for bit in [2]:
  runs = api.runs(path=f"{ENTITY}/{PROJECT}",
                  filters={"$or": [
                          { "$and": [
                            # {"config.quantization_technique": quantization_technique.lower()},
                            # {"config.bit_width": f"{bit}-bit"},
                            {"config.num_calibration_samples": num_samples},
                            {"config.evaluation_dataset": f"{eval_dataset.lower()}{eval_dataset_variant}"},
                              {"config.base_model": "Qwen/Qwen3-14B"},
                            # {"config.base_model": base_model},
                            {"config.enable_thinking": False},
                            {"config.output_type": "multiple_choice"},
                            {"$not": {"config.calibration_language": "WuuChinese"}},
                            {"$not": {"$and": [{"created_at": {"$lt": "2026-03-01"}}, {"config.quantization_technique": "gptq"}]}},
                            {"created_at": {"$gt": "2026-07-01"}},
                          ]
                          },
                          { "$and": [
                            {"config.quantization_technique": "sinq"},
                            # {"config.bit_width": f"{bit}-bit"},
                            {"config.evaluation_dataset": f"{eval_dataset.lower()}{eval_dataset_variant}"},
                            # {"config.base_model": base_model},
                            {"config.enable_thinking": False},
                              {"config.base_model": "Qwen/Qwen3-14B"},
                            {"config.output_type": "multiple_choice"},
                            {"created_at": {"$gt": "2026-07-01"}},
                          ]
                          },
                          { "$and" : [
                              {"config.quantization_technique": "Unquantized"},
                              {"config.evaluation_dataset": f"{eval_dataset.lower()}{eval_dataset_variant}"},
                              # {"config.base_model": base_model},
                              {"config.enable_thinking": False},
                              {"config.base_model": "Qwen/Qwen3-14B"},
                              {"created_at": {"$gt": "2026-07-01"}},
                          ]}
                      ]}
                  )

  visualize_deliver_bar(eval_languages=eval_languages, runs=runs, base_models=["Qwen/Qwen3-14B"], quantization_technique=quantization_technique, bits=["2-bit", "3-bit"], eval_dataset=eval_dataset, custom_title="", display_mean=False, compile=False, vertical_spacing=0.038, horizontal_spacing=0.040, random_guess_score=1/18, write_pdf="massive_14b_barplot_reduced_errorbar_final.pdf")

Pre-AGG


,Base Model,Quantization Technique,Bit Width,Calibration Language,en-US,ta-IN,id-ID,sw-KE,zh-CN
2,Qwen/Qwen3-14B,gptq,2-bit,English,0.113315,0.082381,0.089778,0.083389,0.088097
4,Qwen/Qwen3-14B,gptq,3-bit,English,0.518494,0.428379,0.491258,0.349361,0.471083
3,Qwen/Qwen3-14B,gptq,2-bit,Tamil,0.087088,0.075992,0.080363,0.068594,0.087424
5,Qwen/Qwen3-14B,gptq,3-bit,Tamil,0.561533,0.537660,0.553800,0.411903,0.546066
6,Qwen/Qwen3-14B,gptq,2-bit,Indonesian,0.111298,0.072293,0.107599,0.066577,0.087761
7,Qwen/Qwen3-14B,gptq,3-bit,Indonesian,0.544048,0.464694,0.527236,0.386348,0.508406
8,Qwen/Qwen3-14B,gptq,2-bit,Swahili,0.091123,0.073638,0.093813,0.079691,0.074311
0,Qwen/Qwen3-14B,gptq,3-bit,Swahili,0.571957,0.519166,0.541358,0.459314,0.549428
9,Qwen/Qwen3-14B,gptq,2-bit,Chinese,0.112979,0.084398,0.093141,0.075656,0.112979
1,Qwen/Qwen3-14B,gptq,3-bit,Chinese,0.570948,0.515131,0.533289,0.390720,0.553127


,Base Model,Quantization Technique,Bit Width,Calibration Language,en-US,ta-IN,id-ID,sw-KE,zh-CN


,Base Model,Quantization Technique,Bit Width,Calibration Language,en-US,ta-IN,id-ID,sw-KE,zh-CN
4,Qwen/Qwen3-14B,tacq,2-bit,English,0.488904,0.113652,0.303631,0.140888,0.442502
7,Qwen/Qwen3-14B,tacq,3-bit,English,0.582381,0.486214,0.552455,0.393073,0.561533
6,Qwen/Qwen3-14B,tacq,2-bit,Tamil,0.450235,0.417956,0.387021,0.208137,0.448890
9,Qwen/Qwen3-14B,tacq,3-bit,Tamil,0.594822,0.551110,0.545393,0.432414,0.561870
5,Qwen/Qwen3-14B,tacq,2-bit,Indonesian,0.466039,0.187962,0.463349,0.195360,0.442838
8,Qwen/Qwen3-14B,tacq,3-bit,Indonesian,0.588433,0.525891,0.568258,0.442165,0.565568
0,Qwen/Qwen3-14B,tacq,2-bit,Swahili,0.394418,0.147276,0.354069,0.289173,0.389711
2,Qwen/Qwen3-14B,tacq,3-bit,Swahili,0.579691,0.513786,0.537996,0.445528,0.575656
1,Qwen/Qwen3-14B,tacq,2-bit,Chinese,0.462004,0.120040,0.316073,0.159045,0.458978
3,Qwen/Qwen3-14B,tacq,3-bit,Chinese,0.562542,0.498655,0.531271,0.390383,0.546402


Contrasting Acc and Stderr
GPTQ


...

,Base Model,Quantization Technique,Calibration Language,Bit Width,en-US,ta-IN,id-ID,sw-KE,zh-CN


TACQ


,Base Model,Quantization Technique,Calibration Language,Bit Width,en-US,ta-IN,id-ID,sw-KE,zh-CN
2,Qwen/Qwen3-14B,tacq,English,2-bit,0.488904,0.113652,0.303631,0.140888,0.442502
3,Qwen/Qwen3-14B,tacq,English,3-bit,0.582381,0.486214,0.552455,0.393073,0.561533
8,Qwen/Qwen3-14B,tacq,Tamil,2-bit,0.450235,0.417956,0.387021,0.208137,0.448890
9,Qwen/Qwen3-14B,tacq,Tamil,3-bit,0.594822,0.551110,0.545393,0.432414,0.561870
4,Qwen/Qwen3-14B,tacq,Indonesian,2-bit,0.466039,0.187962,0.463349,0.195360,0.442838
5,Qwen/Qwen3-14B,tacq,Indonesian,3-bit,0.588433,0.525891,0.568258,0.442165,0.565568
6,Qwen/Qwen3-14B,tacq,Swahili,2-bit,0.394418,0.147276,0.354069,0.289173,0.389711
7,Qwen/Qwen3-14B,tacq,Swahili,3-bit,0.579691,0.513786,0.537996,0.445528,0.575656
0,Qwen/Qwen3-14B,tacq,Chinese,2-bit,0.462004,0.120040,0.316073,0.159045,0.458978
1,Qwen/Qwen3-14B,tacq,Chinese,3-bit,0.562542,0.498655,0.531271,0.390383,0.546402


,Base Model,Quantization Technique,Calibration Language,Bit Width,en-US,ta-IN,id-ID,sw-KE,zh-CN


[[0, 'EN'], [1, 'TA'], [2, 'ID'], [3, 'SW'], [4, 'ZH']]
where TA located
[]
[[0, 'EN'], [1, 'TA'], [2, 'ID'], [3, 'SW'], [4, 'ZH']]
where TA located
[]


### XStoryCloze Bar

In [ ]:
eval_languages = ["en", "id", "sw", "zh"]

# Initialize the W&B API
api = Api()

quantization_technique = "GPTQ"
# bit = 8
num_samples = 128
eval_dataset = "xstorycloze"
# base_model = "Qwen/Qwen3-8B"
base_model = "All"

# for base_model in ["Qwen/Qwen3-8B", "meta-llama/Llama-3.1-8B-Instruct", "CohereLabs/aya-expanse-8b"]:
for bit in [3]:
  runs = api.runs(path=f"{ENTITY}/{PROJECT}",
                  filters={"$or": [
                          { "$and": [
                            # {"config.quantization_technique": quantization_technique.lower()},
                            # {"config.bit_width": f"{bit}-bit"},
                            {"config.num_calibration_samples": num_samples},
                            {"$or": [
                                {"$and": [
                                  {"config.evaluation_dataset": f"{eval_dataset.lower()}"},
                                  {"created_at": {"$lt": "2026-07-01"}},
                                ]},
                                {"$and": [
                                  {"$or": [
                                    {"config.evaluation_dataset": f"{eval_dataset.lower()}-1000"},
                                    {"config.evaluation_dataset": f"{eval_dataset.lower()}-2000"},
                                  ]},
                                  {"created_at": {"$gt": "2026-07-01"}},
                                ]},
                            ]},
                            # {"config.base_model": base_model},
                            {"config.enable_thinking": False},
                            {"config.output_type": "multiple_choice"},
                            {"$not": {"config.calibration_language": "WuuChinese"}},
                            {"$not": {"config.calibration_language": "Tamil"}},
                            {"$not": {"$and": [{"created_at": {"$lt": "2026-03-01"}}, {"config.quantization_technique": "gptq"}]}},
                          ]
                          },
                          { "$and": [
                            {"config.quantization_technique": "sinq"},
                            # {"config.bit_width": f"{bit}-bit"},
                            {"config.evaluation_dataset": eval_dataset.lower()},
                            # {"config.base_model": base_model},
                            {"config.enable_thinking": False},
                            {"config.output_type": "multiple_choice"},
                            {"created_at": {"$gt": "2026-07-01"}},
                          ]
                          },
                          { "$and" : [
                              {"config.quantization_technique": "Unquantized"},
                              {"config.evaluation_dataset": eval_dataset.lower()},
                              # {"config.base_model": base_model},
                              {"config.enable_thinking": False},
                              {"created_at": {"$lt": "2026-07-01"}},
                          ]}
                      ]}
                  )

  # visualize_delta_unquantized_deliver_bar(eval_languages=eval_languages, runs=runs, base_model=base_model, quantization_technique=quantization_technique, bits=['2-bit', '3-bit'], eval_dataset=eval_dataset, custom_title="", display_mean=False, compile=True)
  visualize_deliver_bar(eval_languages=eval_languages, runs=runs, base_models=["Qwen/Qwen3-8B", "CohereLabs/aya-expanse-8b"], quantization_technique=quantization_technique, bits=['2-bit', '3-bit', '4-bit'], eval_dataset=eval_dataset, custom_title="", display_mean=False, compile=True, vertical_spacing=0.08, horizontal_spacing=0.025, random_guess_score=1/2, write_pdf="xstorycloze_barplot_sinq_final.pdf")

Pre-AGG


,Base Model,Quantization Technique,Bit Width,Calibration Language,en,id,sw,zh
8,Qwen/Qwen3-8B,gptq,2-bit,English,0.503640,0.453342,0.468564,0.472535
24,Qwen/Qwen3-8B,gptq,2-bit,English,0.497684,0.448048,0.463931,0.467240
25,Qwen/Qwen3-8B,gptq,2-bit,English,0.510258,0.462608,0.463931,0.479815
4,Qwen/Qwen3-8B,gptq,3-bit,English,0.666446,0.596956,0.511582,0.602912
32,Qwen/Qwen3-8B,gptq,3-bit,English,0.690933,0.602250,0.510920,0.608868
33,Qwen/Qwen3-8B,gptq,3-bit,English,0.697551,0.610854,0.508273,0.625414
0,Qwen/Qwen3-8B,gptq,4-bit,English,0.685639,0.617472,0.523494,0.618795
9,Qwen/Qwen3-8B,gptq,2-bit,Indonesian,0.455989,0.483786,0.464593,0.457975
26,Qwen/Qwen3-8B,gptq,2-bit,Indonesian,0.488418,0.473197,0.467240,0.469226
27,Qwen/Qwen3-8B,gptq,2-bit,Indonesian,0.490404,0.469887,0.473858,0.459298


,Base Model,Quantization Technique,Bit Width,Calibration Language,en,id,sw,zh
8,Qwen/Qwen3-8B,slimllm,2-bit,English,0.540702,0.463269,0.481800,0.477167
4,Qwen/Qwen3-8B,slimllm,3-bit,English,0.665784,0.591661,0.505625,0.600265
24,Qwen/Qwen3-8B,slimllm,3-bit,English,0.675711,0.592985,0.513567,0.596294
25,Qwen/Qwen3-8B,slimllm,3-bit,English,0.657842,0.565189,0.489742,0.587690
0,Qwen/Qwen3-8B,slimllm,4-bit,English,0.692257,0.624090,0.524156,0.623428
9,Qwen/Qwen3-8B,slimllm,2-bit,Indonesian,0.510258,0.501655,0.485109,0.491066
5,Qwen/Qwen3-8B,slimllm,3-bit,Indonesian,0.637988,0.591661,0.501655,0.596956
26,Qwen/Qwen3-8B,slimllm,3-bit,Indonesian,0.672402,0.613501,0.512244,0.595632
27,Qwen/Qwen3-8B,slimllm,3-bit,Indonesian,0.653872,0.605559,0.491066,0.595632
1,Qwen/Qwen3-8B,slimllm,4-bit,Indonesian,0.696228,0.635341,0.516214,0.621443


,Base Model,Quantization Technique,Bit Width,Calibration Language,en,id,sw,zh
8,Qwen/Qwen3-8B,tacq,2-bit,English,0.636664,0.477167,0.490404,0.544672
4,Qwen/Qwen3-8B,tacq,3-bit,English,0.682991,0.590999,0.516214,0.610854
24,Qwen/Qwen3-8B,tacq,3-bit,English,0.696889,0.614825,0.524818,0.621443
25,Qwen/Qwen3-8B,tacq,3-bit,English,0.690933,0.595632,0.505625,0.612177
0,Qwen/Qwen3-8B,tacq,4-bit,English,0.682330,0.622105,0.521509,0.618795
9,Qwen/Qwen3-8B,tacq,2-bit,Indonesian,0.622105,0.558570,0.487756,0.551291
5,Qwen/Qwen3-8B,tacq,3-bit,Indonesian,0.677035,0.626737,0.514229,0.620781
26,Qwen/Qwen3-8B,tacq,3-bit,Indonesian,0.681006,0.616148,0.514891,0.616148
27,Qwen/Qwen3-8B,tacq,3-bit,Indonesian,0.674388,0.609530,0.511582,0.613501
1,Qwen/Qwen3-8B,tacq,4-bit,Indonesian,0.707478,0.626075,0.519523,0.618134


Contrasting Acc and Stderr
GPTQ


...

,0
en,2.275976
id,2.000662
sw,1.913302
zh,1.988749


[[0, 'EN'], [2, 'ID'], [3, 'SW'], [4, 'ZH']]
where TA located
['EN' 'ID' 'SW' 'ZH']
mean_ppl


,0
en,2.971542
id,2.587028
sw,2.012574
zh,2.528789


[[0, 'EN'], [2, 'ID'], [3, 'SW'], [4, 'ZH']]
where TA located
['EN' 'ID' 'SW' 'ZH']
mean_ppl


,0
en,3.048974
id,2.703508
sw,2.050960
zh,2.614825


In [ ]:
eval_languages = ["en", "id", "sw", "zh"]

# Initialize the W&B API
api = Api()

quantization_technique = "GPTQ"
# bit = 8
num_samples = 128
eval_dataset = "xstorycloze"
# base_model = "Qwen/Qwen3-8B"
base_model = "All"

# for base_model in ["Qwen/Qwen3-8B", "meta-llama/Llama-3.1-8B-Instruct", "CohereLabs/aya-expanse-8b"]:
for bit in [2]:
  runs = api.runs(path=f"{ENTITY}/{PROJECT}",
                  filters={"$or": [
                          { "$and": [
                            # {"config.quantization_technique": quantization_technique.lower()},
                            # {"config.bit_width": f"{bit}-bit"},
                            {"config.num_calibration_samples": num_samples},
                            {"config.evaluation_dataset": f"{eval_dataset.lower()}"},
                              {"config.base_model": "Qwen/Qwen3-14B"},
                            # {"config.base_model": base_model},
                            {"config.enable_thinking": False},
                            {"config.output_type": "multiple_choice"},
                            {"$not": {"config.calibration_language": "WuuChinese"}},
                            {"$not": {"$and": [{"created_at": {"$lt": "2026-03-01"}}, {"config.quantization_technique": "gptq"}]}},
                            {"created_at": {"$gt": "2026-07-01"}},
                            {"$not": {"config.calibration_language": "Tamil"}},
                          ]
                          },
                          { "$and": [
                            {"config.quantization_technique": "sinq"},
                            # {"config.bit_width": f"{bit}-bit"},
                            {"config.evaluation_dataset": eval_dataset.lower()},
                            # {"config.base_model": base_model},
                            {"config.enable_thinking": False},
                              {"config.base_model": "Qwen/Qwen3-14B"},
                            {"config.output_type": "multiple_choice"},
                            {"created_at": {"$gt": "2026-07-01"}},
                          ]
                          },
                          { "$and" : [
                              {"config.quantization_technique": "Unquantized"},
                              {"config.evaluation_dataset": eval_dataset.lower()},
                              # {"config.base_model": base_model},
                              {"config.enable_thinking": False},
                              {"config.base_model": "Qwen/Qwen3-14B"},
                              {"created_at": {"$gt": "2026-07-01"}},
                          ]}
                      ]}
                  )

  visualize_deliver_bar(eval_languages=eval_languages, runs=runs, base_models=["Qwen/Qwen3-14B"], quantization_technique=quantization_technique, bits=["2-bit", "3-bit"], eval_dataset=eval_dataset, custom_title="", display_mean=False, compile=False, vertical_spacing=0.038, horizontal_spacing=0.040, random_guess_score=0.5, write_pdf="xstorycloze_14b_barplot_reduced_errorbar_final.pdf")

Pre-AGG


,Base Model,Quantization Technique,Bit Width,Calibration Language,en,id,sw,zh
2,Qwen/Qwen3-14B,gptq,2-bit,English,0.517538,0.450695,0.459298,0.489080
4,Qwen/Qwen3-14B,gptq,3-bit,English,0.697551,0.632694,0.486433,0.642621
3,Qwen/Qwen3-14B,gptq,2-bit,Indonesian,0.497022,0.493051,0.484447,0.467902
5,Qwen/Qwen3-14B,gptq,3-bit,Indonesian,0.707478,0.634017,0.521509,0.645930
7,Qwen/Qwen3-14B,gptq,2-bit,Swahili,0.482462,0.459960,0.504964,0.458637
0,Qwen/Qwen3-14B,gptq,3-bit,Swahili,0.673726,0.630046,0.537392,0.634679
6,Qwen/Qwen3-14B,gptq,2-bit,Chinese,0.499007,0.443415,0.479153,0.497022
1,Qwen/Qwen3-14B,gptq,3-bit,Chinese,0.696889,0.628723,0.491727,0.652548


,Base Model,Quantization Technique,Bit Width,Calibration Language,en,id,sw,zh


,Base Model,Quantization Technique,Bit Width,Calibration Language,en,id,sw,zh
4,Qwen/Qwen3-14B,tacq,2-bit,English,0.657842,0.478491,0.497684,0.573792
6,Qwen/Qwen3-14B,tacq,3-bit,English,0.699537,0.630708,0.518862,0.653872
5,Qwen/Qwen3-14B,tacq,2-bit,Indonesian,0.643944,0.585043,0.495698,0.587690
7,Qwen/Qwen3-14B,tacq,3-bit,Indonesian,0.702184,0.636664,0.514229,0.661152
0,Qwen/Qwen3-14B,tacq,2-bit,Swahili,0.624752,0.545996,0.516214,0.563203
2,Qwen/Qwen3-14B,tacq,3-bit,Swahili,0.697551,0.632032,0.533422,0.653210
1,Qwen/Qwen3-14B,tacq,2-bit,Chinese,0.638650,0.479153,0.491066,0.582396
3,Qwen/Qwen3-14B,tacq,3-bit,Chinese,0.690933,0.626737,0.519523,0.647915


Contrasting Acc and Stderr
GPTQ


...

,Base Model,Quantization Technique,Calibration Language,Bit Width,en,id,sw,zh


TACQ


,Base Model,Quantization Technique,Calibration Language,Bit Width,en,id,sw,zh
2,Qwen/Qwen3-14B,tacq,English,2-bit,0.657842,0.478491,0.497684,0.573792
3,Qwen/Qwen3-14B,tacq,English,3-bit,0.699537,0.630708,0.518862,0.653872
4,Qwen/Qwen3-14B,tacq,Indonesian,2-bit,0.643944,0.585043,0.495698,0.587690
5,Qwen/Qwen3-14B,tacq,Indonesian,3-bit,0.702184,0.636664,0.514229,0.661152
6,Qwen/Qwen3-14B,tacq,Swahili,2-bit,0.624752,0.545996,0.516214,0.563203
7,Qwen/Qwen3-14B,tacq,Swahili,3-bit,0.697551,0.632032,0.533422,0.653210
0,Qwen/Qwen3-14B,tacq,Chinese,2-bit,0.638650,0.479153,0.491066,0.582396
1,Qwen/Qwen3-14B,tacq,Chinese,3-bit,0.690933,0.626737,0.519523,0.647915


,Base Model,Quantization Technique,Calibration Language,Bit Width,en,id,sw,zh


[[0, 'EN'], [2, 'ID'], [3, 'SW'], [4, 'ZH']]
where TA located
[]
[[0, 'EN'], [2, 'ID'], [3, 'SW'], [4, 'ZH']]
where TA located
[]


### INCLUDE BAR

In [ ]:
eval_languages = ["Dutch", "German", "Chinese", "Indonesian", "Tamil"]
# eval_languages = ["Dutch", "German", "Chinese", "Indonesian", "Malay", "Tagalog", "Tamil", "Telugu", "Malayalam"]
eval_languages = [lang.lower() for lang in eval_languages]

# Initialize the W&B API
api = Api()

quantization_technique = "ALL"
# bit = 8
num_samples = 128
eval_dataset = "include"
# base_model = "Qwen/Qwen3-8B"
base_model = "All"

for bit in [4]:
  runs = api.runs(path=f"{ENTITY}/{PROJECT}",
                  filters={"$or": [
                          { "$and": [
                            # {"config.quantization_technique": quantization_technique.lower()},
                            # {"config.bit_width": f"{bit}-bit"},
                            {"config.num_calibration_samples": num_samples},
                            {"$or": [
                                {"$and": [
                                  {"config.evaluation_dataset": f"{eval_dataset.lower()}"},
                                  {"created_at": {"$lt": "2026-07-01"}},
                                ]},
                                {"$and": [
                                  {"$or": [
                                    {"config.evaluation_dataset": f"{eval_dataset.lower()}-1000"},
                                    {"config.evaluation_dataset": f"{eval_dataset.lower()}-2000"},
                                  ]},
                                  {"created_at": {"$gt": "2026-07-01"}},
                                ]},
                            ]},
                            # {"config.base_model": base_model},
                            {"config.enable_thinking": False},
                            {"config.output_type": "multiple_choice"},
                            {"$not": {"$and": [{"created_at": {"$lt": "2026-03-01"}}, {"config.quantization_technique": "gptq"}]}},
                            {"$not": {"config.calibration_language": "WuuChinese"}},
                            {"$not": {"config.calibration_language": "Swahili"}},
                            {"$not": {"config.bit_width": "8-bit"}},
                            {"$not": {"config.base_model": "Qwen/Qwen3-1.7B"}},
                            {"$not": {"config.base_model": "Qwen/Qwen3-4B"}},
                          ]
                          },
                          { "$and": [
                            {"config.quantization_technique": "sinq"},
                            # {"config.bit_width": f"{bit}-bit"},
                            {"config.evaluation_dataset": eval_dataset.lower()},
                            # {"config.base_model": base_model},
                            {"config.enable_thinking": False},
                            {"config.output_type": "multiple_choice"},
                            {"created_at": {"$gt": "2026-07-01"}},
                          ]
                          },
                          { "$and" : [
                              {"config.quantization_technique": "Unquantized"},
                              {"config.evaluation_dataset": eval_dataset.lower()},
                              # {"config.base_model": base_model},
                              {"config.enable_thinking": False},
                              {"$not": {"config.base_model": "Qwen/Qwen3-1.7B"}},
                              {"$not": {"config.base_model": "Qwen/Qwen3-4B"}},
                            {"created_at": {"$lt": "2026-07-01"}},
                          ]}
                      ]}
                  )

  visualize_deliver_bar(eval_languages=eval_languages, runs=runs, base_models=["Qwen/Qwen3-8B", "CohereLabs/aya-expanse-8b", "meta-llama/Llama-3.1-8B-Instruct"], quantization_technique=quantization_technique, bits=["2-bit", "3-bit", "4-bit"], eval_dataset=eval_dataset, custom_title="", display_mean=False, compile=False, vertical_spacing=0.080, horizontal_spacing=0.030, write_pdf="include_barplot_sinq_final.pdf")

Pre-AGG


,Base Model,Quantization Technique,Bit Width,Calibration Language,dutch,german,tamil,indonesian,chinese
8,Qwen/Qwen3-8B,gptq,2-bit,English,0.192377,0.280576,0.234545,0.240000,0.255046
36,Qwen/Qwen3-8B,gptq,2-bit,English,0.255898,0.280576,0.249091,0.221818,0.244037
37,Qwen/Qwen3-8B,gptq,2-bit,English,0.206897,0.230216,0.252727,0.225455,0.244037
4,Qwen/Qwen3-8B,gptq,3-bit,English,0.401089,0.323741,0.245455,0.418182,0.526606
44,Qwen/Qwen3-8B,gptq,3-bit,English,0.450091,0.287770,0.261818,0.458182,0.544954
...,...,...,...,...,...,...,...,...,...
23,meta-llama/Llama-3.1-8B-Instruct,gptq,2-bit,Chinese,0.197822,0.309353,0.229091,0.232727,0.264220
58,meta-llama/Llama-3.1-8B-Instruct,gptq,2-bit,Chinese,0.257713,0.287770,0.258182,0.241818,0.262385
59,meta-llama/Llama-3.1-8B-Instruct,gptq,2-bit,Chinese,0.239564,0.280576,0.249091,0.247273,0.227523
19,meta-llama/Llama-3.1-8B-Instruct,gptq,3-bit,Chinese,0.319419,0.287770,0.243636,0.360000,0.357798


,Base Model,Quantization Technique,Bit Width,Calibration Language,dutch,german,tamil,indonesian,chinese
4,Qwen/Qwen3-8B,slimllm,2-bit,English,0.165154,0.330935,0.256364,0.252727,0.245872
33,Qwen/Qwen3-8B,slimllm,2-bit,English,0.186933,0.251799,0.252727,0.263636,0.249541
34,Qwen/Qwen3-8B,slimllm,2-bit,English,0.132486,0.323741,0.245455,0.256364,0.244037
26,Qwen/Qwen3-8B,slimllm,3-bit,English,0.393829,0.388489,0.238182,0.460000,0.590826
41,Qwen/Qwen3-8B,slimllm,3-bit,English,0.528131,0.381295,0.270909,0.523636,0.622018
...,...,...,...,...,...,...,...,...,...
18,meta-llama/Llama-3.1-8B-Instruct,slimllm,2-bit,Chinese,0.312160,0.323741,0.285455,0.223636,0.244037
55,meta-llama/Llama-3.1-8B-Instruct,slimllm,2-bit,Chinese,0.221416,0.244604,0.236364,0.272727,0.249541
56,meta-llama/Llama-3.1-8B-Instruct,slimllm,2-bit,Chinese,0.217786,0.208633,0.258182,0.250909,0.269725
22,meta-llama/Llama-3.1-8B-Instruct,slimllm,3-bit,Chinese,0.352087,0.345324,0.267273,0.316364,0.348624


,Base Model,Quantization Technique,Bit Width,Calibration Language,dutch,german,tamil,indonesian,chinese
3,Qwen/Qwen3-8B,tacq,2-bit,English,0.290381,0.316547,0.263636,0.236364,0.308257
36,Qwen/Qwen3-8B,tacq,2-bit,English,0.259528,0.230216,0.207273,0.278182,0.335780
37,Qwen/Qwen3-8B,tacq,2-bit,English,0.281307,0.215827,0.249091,0.274545,0.359633
25,Qwen/Qwen3-8B,tacq,3-bit,English,0.513612,0.467626,0.250909,0.467273,0.622018
44,Qwen/Qwen3-8B,tacq,3-bit,English,0.528131,0.402878,0.314545,0.532727,0.618349
...,...,...,...,...,...,...,...,...,...
13,meta-llama/Llama-3.1-8B-Instruct,tacq,2-bit,Chinese,0.134301,0.330935,0.245455,0.247273,0.260550
58,meta-llama/Llama-3.1-8B-Instruct,tacq,2-bit,Chinese,0.214156,0.287770,0.249091,0.261818,0.275229
59,meta-llama/Llama-3.1-8B-Instruct,tacq,2-bit,Chinese,0.323049,0.287770,0.296364,0.250909,0.249541
23,meta-llama/Llama-3.1-8B-Instruct,tacq,3-bit,Chinese,0.462795,0.417266,0.290909,0.483636,0.436697


['NL', 'DE', 'TA', 'ID', 'ZH']
Contrasting Acc and Stderr
GPTQ


...

,0
english,0.991019
tamil,1.025455
indonesian,1.047879
chinese,0.977982


[[0, 'EN'], [1, 'TA'], [2, 'ID'], [4, 'ZH']]
where TA located
['EN' 'TA' 'ID' 'ZH']
mean_ppl


,0
english,1.258386
tamil,1.016364
indonesian,1.281818
chinese,1.291743


[[0, 'EN'], [1, 'TA'], [2, 'ID'], [4, 'ZH']]
where TA located
['EN' 'TA' 'ID' 'ZH']
mean_ppl


,0
english,1.747425
tamil,1.354545
indonesian,1.958182
chinese,1.772477


In [ ]:
eval_languages = ["Dutch", "German", "Chinese", "Indonesian", "Tamil"]
# eval_languages = ["Dutch", "German", "Chinese", "Indonesian", "Malay", "Tagalog", "Tamil", "Telugu", "Malayalam"]
eval_languages = [lang.lower() for lang in eval_languages]

# Initialize the W&B API
api = Api()

quantization_technique = "ALL"
# bit = 8
num_samples = 128
eval_dataset = "include"
# base_model = "Qwen/Qwen3-8B"
base_model = "All"

# for base_model in ["Qwen/Qwen3-8B", "meta-llama/Llama-3.1-8B-Instruct", "CohereLabs/aya-expanse-8b"]:
for bit in [2]:
  runs = api.runs(path=f"{ENTITY}/{PROJECT}",
                  filters={"$or": [
                          { "$and": [
                            # {"config.quantization_technique": quantization_technique.lower()},
                            # {"config.bit_width": f"{bit}-bit"},
                            {"config.num_calibration_samples": num_samples},
                            {"config.evaluation_dataset": f"{eval_dataset.lower()}"},
                              {"config.base_model": "Qwen/Qwen3-14B"},
                            # {"config.base_model": base_model},
                            {"config.enable_thinking": False},
                            {"config.output_type": "multiple_choice"},
                            {"$not": {"config.calibration_language": "WuuChinese"}},
                            {"$not": {"$and": [{"created_at": {"$lt": "2026-03-01"}}, {"config.quantization_technique": "gptq"}]}},
                            {"created_at": {"$gt": "2026-07-01"}},
                            {"$not": {"config.calibration_language": "Swahili"}},
                          ]
                          },
                          { "$and": [
                            {"config.quantization_technique": "sinq"},
                            # {"config.bit_width": f"{bit}-bit"},
                            {"config.evaluation_dataset": eval_dataset.lower()},
                            # {"config.base_model": base_model},
                            {"config.enable_thinking": False},
                              {"config.base_model": "Qwen/Qwen3-14B"},
                            {"config.output_type": "multiple_choice"},
                            {"created_at": {"$gt": "2026-07-01"}},
                          ]
                          },
                          { "$and" : [
                              {"config.quantization_technique": "Unquantized"},
                              {"config.evaluation_dataset": eval_dataset.lower()},
                              # {"config.base_model": base_model},
                              {"config.enable_thinking": False},
                              {"config.base_model": "Qwen/Qwen3-14B"},
                              {"created_at": {"$gt": "2026-07-01"}},
                          ]}
                      ]}
                  )

  visualize_deliver_bar(eval_languages=eval_languages, runs=runs, base_models=["Qwen/Qwen3-14B"], quantization_technique=quantization_technique, bits=["2-bit", "3-bit"], eval_dataset=eval_dataset, custom_title="", display_mean=False, compile=False, vertical_spacing=0.038, horizontal_spacing=0.040, random_guess_score=0.25, write_pdf="include_14b_barplot_reduced_errorbar_final.pdf")

Pre-AGG


,Base Model,Quantization Technique,Bit Width,Calibration Language,dutch,german,tamil,indonesian,chinese
1,Qwen/Qwen3-14B,gptq,2-bit,English,0.192377,0.294964,0.229091,0.258182,0.253211
4,Qwen/Qwen3-14B,gptq,3-bit,English,0.678766,0.503597,0.356364,0.654545,0.722936
2,Qwen/Qwen3-14B,gptq,2-bit,Tamil,0.183303,0.287770,0.241818,0.207273,0.240367
5,Qwen/Qwen3-14B,gptq,3-bit,Tamil,0.666062,0.525180,0.407273,0.627273,0.735780
3,Qwen/Qwen3-14B,gptq,2-bit,Indonesian,0.239564,0.258993,0.232727,0.267273,0.264220
6,Qwen/Qwen3-14B,gptq,3-bit,Indonesian,0.678766,0.496403,0.398182,0.656364,0.721101
7,Qwen/Qwen3-14B,gptq,2-bit,Chinese,0.159710,0.258993,0.238182,0.252727,0.249541
0,Qwen/Qwen3-14B,gptq,3-bit,Chinese,0.660617,0.510791,0.358182,0.640000,0.717431


,Base Model,Quantization Technique,Bit Width,Calibration Language,dutch,german,tamil,indonesian,chinese


,Base Model,Quantization Technique,Bit Width,Calibration Language,dutch,german,tamil,indonesian,chinese
2,Qwen/Qwen3-14B,tacq,2-bit,English,0.375681,0.323741,0.256364,0.285455,0.368807
5,Qwen/Qwen3-14B,tacq,3-bit,English,0.709619,0.546763,0.340000,0.672727,0.724771
4,Qwen/Qwen3-14B,tacq,2-bit,Tamil,0.346642,0.330935,0.321818,0.323636,0.381651
6,Qwen/Qwen3-14B,tacq,3-bit,Tamil,0.682396,0.517986,0.410909,0.674545,0.746789
3,Qwen/Qwen3-14B,tacq,2-bit,Indonesian,0.399274,0.395683,0.292727,0.490909,0.418349
7,Qwen/Qwen3-14B,tacq,3-bit,Indonesian,0.713249,0.539568,0.341818,0.690909,0.732110
0,Qwen/Qwen3-14B,tacq,2-bit,Chinese,0.328494,0.266187,0.245455,0.290909,0.467890
1,Qwen/Qwen3-14B,tacq,3-bit,Chinese,0.689655,0.539568,0.365455,0.656364,0.737615


['NL', 'DE', 'TA', 'ID', 'ZH']
Contrasting Acc and Stderr
GPTQ


...

,Base Model,Quantization Technique,Calibration Language,Bit Width,english,tamil,indonesian,chinese


TACQ


,Base Model,Quantization Technique,Calibration Language,Bit Width,english,tamil,indonesian,chinese
2,Qwen/Qwen3-14B,tacq,English,2-bit,0.349711,0.256364,0.285455,0.368807
3,Qwen/Qwen3-14B,tacq,English,3-bit,0.628191,0.340000,0.672727,0.724771
6,Qwen/Qwen3-14B,tacq,Tamil,2-bit,0.338789,0.321818,0.323636,0.381651
7,Qwen/Qwen3-14B,tacq,Tamil,3-bit,0.600191,0.410909,0.674545,0.746789
4,Qwen/Qwen3-14B,tacq,Indonesian,2-bit,0.397479,0.292727,0.490909,0.418349
5,Qwen/Qwen3-14B,tacq,Indonesian,3-bit,0.626408,0.341818,0.690909,0.732110
0,Qwen/Qwen3-14B,tacq,Chinese,2-bit,0.297340,0.245455,0.290909,0.467890
1,Qwen/Qwen3-14B,tacq,Chinese,3-bit,0.614612,0.365455,0.656364,0.737615


,Base Model,Quantization Technique,Calibration Language,Bit Width,english,tamil,indonesian,chinese


[[0, 'EN'], [1, 'TA'], [2, 'ID'], [4, 'ZH']]
where TA located
[]
[[0, 'EN'], [1, 'TA'], [2, 'ID'], [4, 'ZH']]
where TA located
[]


### GlobalMMLULite Bar

In [ ]:
eval_languages = ["en", "zh", "id", "sw"]

# Initialize the W&B API
api = Api()

quantization_technique = "ALL"
# bit = 8
num_samples = 128
eval_dataset = "globalmmlulite"
# base_model = "Qwen/Qwen3-8B"
base_model = "All"

for bit in [4]:
  runs = api.runs(path=f"{ENTITY}/{PROJECT}",
                  filters={"$or": [
                          { "$and": [
                            # {"config.quantization_technique": quantization_technique.lower()},
                            # {"config.bit_width": f"{bit}-bit"},
                            {"config.num_calibration_samples": num_samples},
                            {"$or": [
                                {"$and": [
                                  {"config.evaluation_dataset": f"{eval_dataset.lower()}"},
                                  {"created_at": {"$lt": "2026-07-01"}},
                                ]},
                                {"$and": [
                                  {"$or": [
                                    {"config.evaluation_dataset": f"{eval_dataset.lower()}-1000"},
                                    {"config.evaluation_dataset": f"{eval_dataset.lower()}-2000"},
                                  ]},
                                  {"created_at": {"$gt": "2026-07-01"}},
                                ]},
                            ]},
                            # {"config.base_model": base_model},
                            {"config.enable_thinking": False},
                            {"config.output_type": "multiple_choice"},
                            {"$not": {"$and": [{"created_at": {"$lt": "2026-03-01"}}, {"config.quantization_technique": "gptq"}]}},
                            {"$not": {"config.calibration_language": "WuuChinese"}},
                            {"$not": {"config.calibration_language": "Tamil"}},
                            {"$not": {"config.bit_width": "8-bit"}},
                            {"$not": {"config.base_model": "Qwen/Qwen3-1.7B"}},
                            {"$not": {"config.base_model": "Qwen/Qwen3-4B"}},
                          ]
                          },
                          { "$and": [
                            {"config.quantization_technique": "sinq"},
                            # {"config.bit_width": f"{bit}-bit"},
                            {"config.evaluation_dataset": eval_dataset.lower()},
                            # {"config.base_model": base_model},
                            {"config.enable_thinking": False},
                            {"config.output_type": "multiple_choice"},
                            {"created_at": {"$gt": "2026-07-01"}},
                          ]
                          },
                          { "$and" : [
                              {"config.quantization_technique": "Unquantized"},
                              {"config.evaluation_dataset": eval_dataset.lower()},
                              # {"config.base_model": base_model},
                              {"config.enable_thinking": False},
                              {"$not": {"config.base_model": "Qwen/Qwen3-1.7B"}},
                              {"$not": {"config.base_model": "Qwen/Qwen3-4B"}},
                            {"created_at": {"$lt": "2026-07-01"}},
                          ]}
                      ]}
                  )

  visualize_deliver_bar(eval_languages=eval_languages, runs=runs, base_models=["Qwen/Qwen3-8B", "CohereLabs/aya-expanse-8b", "meta-llama/Llama-3.1-8B-Instruct"], quantization_technique=quantization_technique, bits=["2-bit", "3-bit", "4-bit"], eval_dataset=eval_dataset, custom_title="", display_mean=False, compile=False, vertical_spacing=0.08, horizontal_spacing=0.035, write_pdf="globalmmlulite_barplot_sinq_final.pdf")

Pre-AGG


,Base Model,Quantization Technique,Bit Width,Calibration Language,en,id,sw,zh
4,Qwen/Qwen3-8B,gptq,2-bit,English,0.2550,0.2425,0.2650,0.2450
36,Qwen/Qwen3-8B,gptq,2-bit,English,0.2675,0.2700,0.2325,0.2350
37,Qwen/Qwen3-8B,gptq,2-bit,English,0.2350,0.2725,0.2400,0.2350
11,Qwen/Qwen3-8B,gptq,3-bit,English,0.5050,0.3600,0.2725,0.4225
44,Qwen/Qwen3-8B,gptq,3-bit,English,0.5850,0.4875,0.2950,0.5225
...,...,...,...,...,...,...,...,...
20,meta-llama/Llama-3.1-8B-Instruct,gptq,2-bit,Chinese,0.2400,0.2325,0.2325,0.2800
58,meta-llama/Llama-3.1-8B-Instruct,gptq,2-bit,Chinese,0.2625,0.2400,0.2325,0.2175
59,meta-llama/Llama-3.1-8B-Instruct,gptq,2-bit,Chinese,0.2350,0.2275,0.2475,0.2400
16,meta-llama/Llama-3.1-8B-Instruct,gptq,3-bit,Chinese,0.5025,0.3175,0.2725,0.4250


,Base Model,Quantization Technique,Bit Width,Calibration Language,en,id,sw,zh
4,Qwen/Qwen3-8B,slimllm,2-bit,English,0.2350,0.2400,0.2300,0.2400
36,Qwen/Qwen3-8B,slimllm,2-bit,English,0.2450,0.2775,0.2975,0.2450
37,Qwen/Qwen3-8B,slimllm,2-bit,English,0.2450,0.2650,0.2350,0.2300
30,Qwen/Qwen3-8B,slimllm,3-bit,English,0.5400,0.3425,0.2675,0.4950
44,Qwen/Qwen3-8B,slimllm,3-bit,English,0.6600,0.5525,0.3200,0.5525
...,...,...,...,...,...,...,...,...
13,meta-llama/Llama-3.1-8B-Instruct,slimllm,2-bit,Chinese,0.2800,0.2600,0.2525,0.2575
58,meta-llama/Llama-3.1-8B-Instruct,slimllm,2-bit,Chinese,0.2625,0.2325,0.2575,0.2775
59,meta-llama/Llama-3.1-8B-Instruct,slimllm,2-bit,Chinese,0.2625,0.2525,0.2875,0.2725
17,meta-llama/Llama-3.1-8B-Instruct,slimllm,3-bit,Chinese,0.3975,0.3550,0.2975,0.3775


,Base Model,Quantization Technique,Bit Width,Calibration Language,en,id,sw,zh
3,Qwen/Qwen3-8B,tacq,2-bit,English,0.3350,0.2550,0.2575,0.3625
36,Qwen/Qwen3-8B,tacq,2-bit,English,0.3375,0.2600,0.2575,0.2975
37,Qwen/Qwen3-8B,tacq,2-bit,English,0.3275,0.2675,0.2775,0.3150
32,Qwen/Qwen3-8B,tacq,3-bit,English,0.6175,0.4975,0.2700,0.6075
44,Qwen/Qwen3-8B,tacq,3-bit,English,0.7000,0.5850,0.3600,0.6275
...,...,...,...,...,...,...,...,...
15,meta-llama/Llama-3.1-8B-Instruct,tacq,2-bit,Chinese,0.2425,0.2500,0.2275,0.2475
58,meta-llama/Llama-3.1-8B-Instruct,tacq,2-bit,Chinese,0.2875,0.2800,0.2850,0.3100
59,meta-llama/Llama-3.1-8B-Instruct,tacq,2-bit,Chinese,0.2575,0.3125,0.2800,0.2925
19,meta-llama/Llama-3.1-8B-Instruct,tacq,3-bit,Chinese,0.5450,0.4400,0.3200,0.4125


Contrasting Acc and Stderr
GPTQ


...

,0
en,1.014167
id,1.026667
sw,1.005000
zh,0.985000


[[0, 'EN'], [2, 'ID'], [3, 'SW'], [4, 'ZH']]
where TA located
['EN' 'ID' 'SW' 'ZH']
mean_ppl


,0
en,1.4900
id,1.3025
sw,1.0550
zh,1.3100


[[0, 'EN'], [2, 'ID'], [3, 'SW'], [4, 'ZH']]
where TA located
['EN' 'ID' 'SW' 'ZH']
mean_ppl


,0
en,2.4375
id,2.0825
sw,1.5200
zh,1.9950


In [ ]:
eval_languages = ["en", "zh", "id", "sw"]

# Initialize the W&B API
api = Api()

quantization_technique = "ALL"
# bit = 8
num_samples = 128
eval_dataset = "globalmmlulite"
# base_model = "Qwen/Qwen3-8B"
base_model = "All"

# for base_model in ["Qwen/Qwen3-8B", "meta-llama/Llama-3.1-8B-Instruct", "CohereLabs/aya-expanse-8b"]:
for bit in [2]:
  runs = api.runs(path=f"{ENTITY}/{PROJECT}",
                  filters={"$or": [
                          { "$and": [
                            # {"config.quantization_technique": quantization_technique.lower()},
                            # {"config.bit_width": f"{bit}-bit"},
                            {"config.num_calibration_samples": num_samples},
                            {"config.evaluation_dataset": f"{eval_dataset.lower()}"},
                              {"config.base_model": "Qwen/Qwen3-14B"},
                            # {"config.base_model": base_model},
                            {"config.enable_thinking": False},
                            {"config.output_type": "multiple_choice"},
                            {"$not": {"config.calibration_language": "WuuChinese"}},
                            {"$not": {"$and": [{"created_at": {"$lt": "2026-03-01"}}, {"config.quantization_technique": "gptq"}]}},
                            {"created_at": {"$gt": "2026-07-01"}},
                            {"$not": {"config.calibration_language": "Tamil"}},
                          ]
                          },
                          { "$and": [
                            {"config.quantization_technique": "sinq"},
                            # {"config.bit_width": f"{bit}-bit"},
                            {"config.evaluation_dataset": eval_dataset.lower()},
                            # {"config.base_model": base_model},
                            {"config.enable_thinking": False},
                              {"config.base_model": "Qwen/Qwen3-14B"},
                            {"config.output_type": "multiple_choice"},
                            {"created_at": {"$gt": "2026-07-01"}},
                          ]
                          },
                          { "$and" : [
                              {"config.quantization_technique": "Unquantized"},
                              {"config.evaluation_dataset": eval_dataset.lower()},
                              # {"config.base_model": base_model},
                              {"config.enable_thinking": False},
                              {"config.base_model": "Qwen/Qwen3-14B"},
                              {"created_at": {"$gt": "2026-07-01"}},
                          ]}
                      ]}
                  )

  visualize_deliver_bar(eval_languages=eval_languages, runs=runs, base_models=["Qwen/Qwen3-14B"], quantization_technique=quantization_technique, bits=["2-bit", "3-bit"], eval_dataset=eval_dataset, custom_title="", display_mean=False, compile=False, vertical_spacing=0.038, horizontal_spacing=0.040, random_guess_score=0.25, write_pdf="globalmmlulite_14b_barplot_reduced_errorbar_final.pdf")

Pre-AGG


,Base Model,Quantization Technique,Bit Width,Calibration Language,en,id,sw,zh
2,Qwen/Qwen3-14B,gptq,2-bit,English,0.2125,0.2525,0.2275,0.2500
4,Qwen/Qwen3-14B,gptq,3-bit,English,0.7200,0.6375,0.3925,0.6575
3,Qwen/Qwen3-14B,gptq,2-bit,Indonesian,0.2450,0.2300,0.2750,0.1725
5,Qwen/Qwen3-14B,gptq,3-bit,Indonesian,0.7250,0.5950,0.3875,0.6500
6,Qwen/Qwen3-14B,gptq,2-bit,Swahili,0.2400,0.2675,0.2550,0.2425
0,Qwen/Qwen3-14B,gptq,3-bit,Swahili,0.7325,0.6475,0.3975,0.6825
7,Qwen/Qwen3-14B,gptq,2-bit,Chinese,0.2275,0.2425,0.2750,0.2175
1,Qwen/Qwen3-14B,gptq,3-bit,Chinese,0.7200,0.6100,0.3425,0.5950


,Base Model,Quantization Technique,Bit Width,Calibration Language,en,id,sw,zh


,Base Model,Quantization Technique,Bit Width,Calibration Language,en,id,sw,zh
4,Qwen/Qwen3-14B,tacq,2-bit,English,0.4575,0.2750,0.2350,0.3225
6,Qwen/Qwen3-14B,tacq,3-bit,English,0.7225,0.6000,0.3300,0.6675
5,Qwen/Qwen3-14B,tacq,2-bit,Indonesian,0.4475,0.3875,0.2675,0.4300
7,Qwen/Qwen3-14B,tacq,3-bit,Indonesian,0.7375,0.6500,0.3950,0.6650
0,Qwen/Qwen3-14B,tacq,2-bit,Swahili,0.3850,0.2925,0.2650,0.3275
2,Qwen/Qwen3-14B,tacq,3-bit,Swahili,0.7350,0.6625,0.3925,0.6625
1,Qwen/Qwen3-14B,tacq,2-bit,Chinese,0.4200,0.2900,0.2500,0.3950
3,Qwen/Qwen3-14B,tacq,3-bit,Chinese,0.7125,0.6125,0.3625,0.6675


Contrasting Acc and Stderr
GPTQ


...

,Base Model,Quantization Technique,Calibration Language,Bit Width,en,id,sw,zh


TACQ


,Base Model,Quantization Technique,Calibration Language,Bit Width,en,id,sw,zh
2,Qwen/Qwen3-14B,tacq,English,2-bit,0.4575,0.2750,0.2350,0.3225
3,Qwen/Qwen3-14B,tacq,English,3-bit,0.7225,0.6000,0.3300,0.6675
4,Qwen/Qwen3-14B,tacq,Indonesian,2-bit,0.4475,0.3875,0.2675,0.4300
5,Qwen/Qwen3-14B,tacq,Indonesian,3-bit,0.7375,0.6500,0.3950,0.6650
6,Qwen/Qwen3-14B,tacq,Swahili,2-bit,0.3850,0.2925,0.2650,0.3275
7,Qwen/Qwen3-14B,tacq,Swahili,3-bit,0.7350,0.6625,0.3925,0.6625
0,Qwen/Qwen3-14B,tacq,Chinese,2-bit,0.4200,0.2900,0.2500,0.3950
1,Qwen/Qwen3-14B,tacq,Chinese,3-bit,0.7125,0.6125,0.3625,0.6675


,Base Model,Quantization Technique,Calibration Language,Bit Width,en,id,sw,zh


[[0, 'EN'], [2, 'ID'], [3, 'SW'], [4, 'ZH']]
where TA located
[]
[[0, 'EN'], [2, 'ID'], [3, 'SW'], [4, 'ZH']]
where TA located
[]


### MGSM Bar

In [ ]:
# eval_languages = ["eng_Latn", "tam_Taml", "ind_Latn", "swh_Latn", "zho_Hans"]
eval_languages = ["en", "ta", "sw", "zh"]

# Initialize the W&B API
api = Api()

quantization_technique = "ALL"
# bit = 8
num_samples = 128
eval_dataset = "mgsm"
# base_model = "Qwen/Qwen3-8B"
base_model = "All"
strict_answer = True

# for base_model in ["Qwen/Qwen3-8B", "meta-llama/Llama-3.1-8B-Instruct", "CohereLabs/aya-expanse-8b"]:
for bit in [2]:
  runs = api.runs(path=f"{ENTITY}/{PROJECT}",
                  filters={"$or": [
                          { "$and": [
                            # {"config.quantization_technique": quantization_technique.lower()},
                            {"$not": {"config.bit_width": f"2-bit"}},
                            {"config.num_calibration_samples": num_samples},
                            {"config.evaluation_dataset": f"{eval_dataset.lower()}"},
                            {"$not": {"config.base_model": "Qwen/Qwen3-14B"}},
                            {"config.enable_thinking": True},
                            # {"config.output_type": "generate_until"},
                            {"$not": {"config.calibration_language": "WuuChinese"}},
                            # {"$not": {"$and": [{"created_at": {"$lt": "2026-03-01"}}, {"config.quantization_technique": "gptq"}]}},
                            {"created_at": {"$gt": "2026-07-01"}},
                          ]
                          },
                          { "$and": [
                            {"config.quantization_technique": "sinq"},
                            {"$not": {"config.bit_width": f"2-bit"}},
                            {"config.evaluation_dataset": eval_dataset.lower()},
                            {"$not": {"config.base_model": "Qwen/Qwen3-14B"}},
                            {"config.enable_thinking": True},
                            # {"config.output_type": "generate_until"},
                            {"created_at": {"$gt": "2026-07-01"}},
                          ]
                          },
                          { "$and" : [
                              {"config.quantization_technique": "Unquantized"},
                              {"config.evaluation_dataset": eval_dataset.lower()},
                            {"$not": {"config.base_model": "Qwen/Qwen3-14B"}},
                              {"config.enable_thinking": True},
                              {"created_at": {"$gt": "2026-07-01"}},
                          ]}
                      ]}
                  )
  print(len(runs))

  # visualize_delta_unquantized_globalmmlulite(eval_languages=eval_languages, runs=runs, base_model=base_model, quantization_technique=quantization_technique, bit_width=bit, eval_dataset=eval_dataset, custom_title="")
  # visualize_delta_unquantized_deliver_bar(eval_languages=eval_languages, runs=runs, base_model=base_model, quantization_technique=quantization_technique, bits=["2-bit", "3-bit", "4-bit"], eval_dataset=eval_dataset, custom_title="", display_mean=False, compile=False)
  visualize_deliver_bar(eval_languages=eval_languages, runs=runs, base_models=["Qwen/Qwen3-8B", "CohereLabs/aya-expanse-8b", "meta-llama/Llama-3.1-8B-Instruct"], quantization_technique=quantization_technique, bits=["3-bit", "4-bit"], eval_dataset=eval_dataset, custom_title="", display_mean=False, generate_until=True, compile=False, vertical_spacing=0.07, horizontal_spacing=0.030, random_guess_score=0.0, strict_answer=strict_answer, write_pdf=f"mgsm_barplot_errorbar_{'strict' if strict_answer else 'flexible'}_final.pdf")

79
Pre-AGG


,Base Model,Quantization Technique,Bit Width,Calibration Language,en,ta,sw,zh
4,Qwen/Qwen3-8B,gptq,3-bit,English,0.444,0.124,0.004,0.408
0,Qwen/Qwen3-8B,gptq,4-bit,English,0.824,0.496,0.232,0.840
5,Qwen/Qwen3-8B,gptq,3-bit,Tamil,0.456,0.288,0.004,0.260
1,Qwen/Qwen3-8B,gptq,4-bit,Tamil,0.744,0.628,0.028,0.820
6,Qwen/Qwen3-8B,gptq,3-bit,Swahili,0.252,0.060,0.000,0.468
2,Qwen/Qwen3-8B,gptq,4-bit,Swahili,0.788,0.520,0.080,0.784
7,Qwen/Qwen3-8B,gptq,3-bit,Chinese,0.256,0.048,0.008,0.472
3,Qwen/Qwen3-8B,gptq,4-bit,Chinese,0.748,0.544,0.040,0.864
8,CohereLabs/aya-expanse-8b,gptq,3-bit,English,0.140,0.000,0.004,0.008
10,CohereLabs/aya-expanse-8b,gptq,4-bit,English,0.528,0.000,0.004,0.252


,Base Model,Quantization Technique,Bit Width,Calibration Language,en,ta,sw,zh
4,Qwen/Qwen3-8B,slimllm,3-bit,English,0.376,0.012,0.008,0.496
0,Qwen/Qwen3-8B,slimllm,4-bit,English,0.760,0.648,0.012,0.808
5,Qwen/Qwen3-8B,slimllm,3-bit,Tamil,0.344,0.360,0.020,0.540
1,Qwen/Qwen3-8B,slimllm,4-bit,Tamil,0.776,0.564,0.092,0.816
6,Qwen/Qwen3-8B,slimllm,3-bit,Swahili,0.704,0.064,0.028,0.512
2,Qwen/Qwen3-8B,slimllm,4-bit,Swahili,0.800,0.652,0.056,0.796
7,Qwen/Qwen3-8B,slimllm,3-bit,Chinese,0.712,0.036,0.004,0.700
3,Qwen/Qwen3-8B,slimllm,4-bit,Chinese,0.816,0.532,0.012,0.788
16,CohereLabs/aya-expanse-8b,slimllm,3-bit,English,0.124,0.000,0.000,0.008
19,CohereLabs/aya-expanse-8b,slimllm,4-bit,English,0.568,0.004,0.004,0.076


,Base Model,Quantization Technique,Bit Width,Calibration Language,en,ta,sw,zh
4,Qwen/Qwen3-8B,tacq,3-bit,English,0.760,0.184,0.004,0.812
0,Qwen/Qwen3-8B,tacq,4-bit,English,0.816,0.652,0.080,0.828
5,Qwen/Qwen3-8B,tacq,3-bit,Tamil,0.724,0.484,0.000,0.776
1,Qwen/Qwen3-8B,tacq,4-bit,Tamil,0.780,0.664,0.064,0.844
6,Qwen/Qwen3-8B,tacq,3-bit,Swahili,0.680,0.420,0.008,0.724
2,Qwen/Qwen3-8B,tacq,4-bit,Swahili,0.776,0.620,0.140,0.792
7,Qwen/Qwen3-8B,tacq,3-bit,Chinese,0.788,0.176,0.000,0.772
3,Qwen/Qwen3-8B,tacq,4-bit,Chinese,0.784,0.600,0.020,0.840
16,CohereLabs/aya-expanse-8b,tacq,3-bit,English,0.328,0.008,0.008,0.032
20,CohereLabs/aya-expanse-8b,tacq,4-bit,English,0.408,0.004,0.000,0.268


Contrasting Acc and Stderr
GPTQ


...

,0
en,0.952
ta,0.012
sw,0.012
zh,0.292


[[0, 'EN'], [1, 'TA'], [3, 'SW'], [4, 'ZH']]
where TA located
['EN' 'TA' 'SW' 'ZH']
mean_ppl


,0
en,0.088
ta,0.016
sw,0.004
zh,0.032


[[0, 'EN'], [1, 'TA'], [3, 'SW'], [4, 'ZH']]
where TA located
['EN' 'TA' 'SW' 'ZH']
mean_ppl


,0
en,2.124
ta,0.240
sw,0.304
zh,0.704


In [7]:
# eval_languages = ["eng_Latn", "tam_Taml", "ind_Latn", "swh_Latn", "zho_Hans"]
eval_languages = ["en", "ta", "sw", "zh"]

# Initialize the W&B API
api = Api()

quantization_technique = "ALL"
# bit = 8
num_samples = 128
eval_dataset = "mgsm"
# base_model = "Qwen/Qwen3-8B"
base_model = "All"
strict_answer = True

# for base_model in ["Qwen/Qwen3-8B", "meta-llama/Llama-3.1-8B-Instruct", "CohereLabs/aya-expanse-8b"]:
for bit in [2]:
  runs = api.runs(path=f"{ENTITY}/{PROJECT}",
                  filters={"$or": [
                          { "$and": [
                            # {"config.quantization_technique": quantization_technique.lower()},
                            # {"config.bit_width": f"{bit}-bit"},
                            {"config.num_calibration_samples": num_samples},
                            {"config.evaluation_dataset": f"{eval_dataset.lower()}"},
                            {"config.base_model": "Qwen/Qwen3-14B"},
                            {"config.enable_thinking": True},
                            # {"config.output_type": "generate_until"},
                            {"$not": {"config.calibration_language": "Indonesian"}},
                            {"$not": {"config.calibration_language": "WuuChinese"}},
                            # {"$not": {"$and": [{"created_at": {"$lt": "2026-03-01"}}, {"config.quantization_technique": "gptq"}]}},
                            {"created_at": {"$gt": "2026-07-01"}},
                          ]
                          },
                          { "$and": [
                            {"config.quantization_technique": "sinq"},
                            # {"config.bit_width": f"{bit}-bit"},
                            {"config.evaluation_dataset": eval_dataset.lower()},
                            {"config.base_model": "Qwen/Qwen3-14B"},
                            {"config.enable_thinking": True},
                            # {"config.output_type": "generate_until"},
                            {"created_at": {"$gt": "2026-07-01"}},
                          ]
                          },
                          { "$and" : [
                              {"config.quantization_technique": "Unquantized"},
                              {"config.evaluation_dataset": eval_dataset.lower()},
                            {"config.base_model": "Qwen/Qwen3-14B"},
                              {"config.enable_thinking": True},
                              {"created_at": {"$gt": "2026-07-01"}},
                          ]}
                      ]}
                  )
  print(len(runs))

  # visualize_delta_unquantized_globalmmlulite(eval_languages=eval_languages, runs=runs, base_model=base_model, quantization_technique=quantization_technique, bit_width=bit, eval_dataset=eval_dataset, custom_title="")
  # visualize_delta_unquantized_deliver_bar(eval_languages=eval_languages, runs=runs, base_model=base_model, quantization_technique=quantization_technique, bits=["2-bit", "3-bit", "4-bit"], eval_dataset=eval_dataset, custom_title="", display_mean=False, compile=False)
  visualize_deliver_bar(eval_languages=eval_languages, runs=runs, base_models=["Qwen/Qwen3-14B"], quantization_technique=quantization_technique, bits=["2-bit", "3-bit"], eval_dataset=eval_dataset, custom_title="", display_mean=False, generate_until=True, compile=False, vertical_spacing=0.040, horizontal_spacing=0.04, random_guess_score=0.0, strict_answer=strict_answer, write_pdf=f"mgsm_14b_barplot_errorbar_{'strict' if strict_answer else 'flexible'}_final.pdf")

19
Pre-AGG


,Base Model,Quantization Technique,Bit Width,Calibration Language,en,ta,sw,zh
5,Qwen/Qwen3-14B,gptq,2-bit,English,0.000,0.000,0.000,0.000
3,Qwen/Qwen3-14B,gptq,3-bit,English,0.876,0.240,0.100,0.808
4,Qwen/Qwen3-14B,gptq,2-bit,Tamil,0.000,0.000,0.000,0.000
2,Qwen/Qwen3-14B,gptq,3-bit,Tamil,0.908,0.488,0.240,0.856
7,Qwen/Qwen3-14B,gptq,2-bit,Swahili,0.000,0.000,0.000,0.000
1,Qwen/Qwen3-14B,gptq,3-bit,Swahili,0.740,0.552,0.216,0.872
0,Qwen/Qwen3-14B,gptq,2-bit,Chinese,0.000,0.000,0.000,0.000
6,Qwen/Qwen3-14B,gptq,3-bit,Chinese,0.916,0.468,0.272,0.832


,Base Model,Quantization Technique,Bit Width,Calibration Language,en,ta,sw,zh


,Base Model,Quantization Technique,Bit Width,Calibration Language,en,ta,sw,zh
4,Qwen/Qwen3-14B,tacq,2-bit,English,0.248,0.000,0.000,0.112
6,Qwen/Qwen3-14B,tacq,3-bit,English,0.872,0.448,0.384,0.860
5,Qwen/Qwen3-14B,tacq,2-bit,Tamil,0.024,0.016,0.000,0.008
7,Qwen/Qwen3-14B,tacq,3-bit,Tamil,0.948,0.784,0.416,0.804
0,Qwen/Qwen3-14B,tacq,2-bit,Swahili,0.020,0.000,0.000,0.040
2,Qwen/Qwen3-14B,tacq,3-bit,Swahili,0.876,0.588,0.512,0.820
1,Qwen/Qwen3-14B,tacq,2-bit,Chinese,0.028,0.000,0.000,0.148
3,Qwen/Qwen3-14B,tacq,3-bit,Chinese,0.848,0.632,0.428,0.868


Contrasting Acc and Stderr
GPTQ


...

,Base Model,Quantization Technique,Calibration Language,Bit Width,en,ta,sw,zh


TACQ


,Base Model,Quantization Technique,Calibration Language,Bit Width,en,ta,sw,zh
2,Qwen/Qwen3-14B,tacq,English,2-bit,0.248,0.000,0.000,0.112
3,Qwen/Qwen3-14B,tacq,English,3-bit,0.872,0.448,0.384,0.860
6,Qwen/Qwen3-14B,tacq,Tamil,2-bit,0.024,0.016,0.000,0.008
7,Qwen/Qwen3-14B,tacq,Tamil,3-bit,0.948,0.784,0.416,0.804
4,Qwen/Qwen3-14B,tacq,Swahili,2-bit,0.020,0.000,0.000,0.040
5,Qwen/Qwen3-14B,tacq,Swahili,3-bit,0.876,0.588,0.512,0.820
0,Qwen/Qwen3-14B,tacq,Chinese,2-bit,0.028,0.000,0.000,0.148
1,Qwen/Qwen3-14B,tacq,Chinese,3-bit,0.848,0.632,0.428,0.868


,Base Model,Quantization Technique,Calibration Language,Bit Width,en,ta,sw,zh


[[0, 'EN'], [1, 'TA'], [3, 'SW'], [4, 'ZH']]
where TA located
[]
[[0, 'EN'], [1, 'TA'], [3, 'SW'], [4, 'ZH']]
where TA located
[]
